# CS781 Phase 1 · Efficient causal Transformer evaluation

Evaluate the saved `efficient_001_mixed217m` weights at 300,011,520 applied input positions.
All decoding settings use the same frozen generation examples from `prep_002_mixed16k`.
The recorded run uses one CUDA device, generation batches of 4, seed 456,
and a 1,024-token cap for each 128-word continuation.

## Recorded results on the balanced validation set

| Decoding | BERTScore F1 | ROUGE-L F1 | BLEU-4 | Incomplete / 100 |
|---|---:|---:|---:|---:|
| Top-p 0.95, T=1 | 0.839827 | 0.244141 | 0.141416 | 0 |

The model uses a bounded per-layer KV cache during generation. The selected top-p setting completed all 100 balanced examples and produced an official-format test archive.

## Run setup

Open this notebook in local Jupyter with a CUDA-enabled environment. Point `PREP_DIR` and
`RUN_DIR` in the input section at the extracted preparation and run folders. Run through
held-out generation and scoring before selecting an official test export. The output root
is relative to the notebook kernel's working directory. The first BERTScore run downloads
the scorer's encoder.


## 1. Local environment

Use a Jupyter kernel with CUDA-enabled PyTorch, NumPy, pandas, `ijson`, and the course scorer's BERTScore dependencies. The recorded environment used Python 3.12, PyTorch 2.11 with CUDA 12.8, NumPy 2.1.3 and BERTScore 0.3.13.

For a compatible CUDA 12.8 environment, install these packages and restart the kernel before loading the model.

```bash
python -m pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
python -m pip install numpy==2.1.3 bert-score==0.3.13 transformers==5.16.1 huggingface-hub==1.29.0 pandas==2.2.3 pytz ijson
```


## 2. Workspace and CUDA check

Use the current working directory for the evaluation code and artifacts. Run a small CUDA operation before loading the checkpoint.


In [1]:
from pathlib import Path
import sys, os, json, pickle, shutil, time, math, gc, platform
from importlib.metadata import version
import numpy as np

BASE_DIR = Path.cwd().resolve()
SRC_DIR = BASE_DIR / 'architecture_src_efficient_local_v1'
OUTPUT_ROOT = BASE_DIR / 'efficient_local_evaluation_output'
ROOT = OUTPUT_ROOT / 'cs781_phase1'
SRC_DIR.mkdir(parents=True, exist_ok=True)
ROOT.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
assert torch.cuda.is_available(), 'CUDA is unavailable in this Jupyter kernel. Check the torch installation and NVIDIA driver.'
DEVICE = torch.device('cuda:0')
print('Jupyter Python:', sys.executable)
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), 'GiB')
print('Compute capability:', torch.cuda.get_device_capability(0))
probe = torch.ones((32, 32), device=DEVICE)
assert (probe @ probe).sum().item() == 32**3
del probe
torch.cuda.empty_cache()
print('CUDA execution passed. Outputs:', OUTPUT_ROOT)


Jupyter Python: C:\Users\ashu7\AppData\Local\Programs\Python\Python312\python.exe
PyTorch: 2.11.0+cu128 | CUDA build: 12.8
GPU: NVIDIA GeForce RTX 5060 Laptop GPU | VRAM: 7.93 GiB
Compute capability: (12, 0)
CUDA execution passed. Outputs: C:\Users\ashu7\Downloads\efficient_local_evaluation_output


## 3. File and checksum utilities

Write JSON and checkpoint metadata atomically, hash artifacts and package saved run files.


In [2]:
_module_text = r'''"""Small file and JSON helpers used by the Phase-1 notebooks."""
import hashlib
import json
import os
import pickle
import zipfile
from pathlib import Path


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def atomic_pickle(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def checked_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != value:
        raise ValueError(f"Saved settings differ: {path}. Use a new experiment ID.")
    atomic_json(path, value)


def write_source(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw = content.encode("utf-8") if isinstance(content, str) else content
    if path.exists():
        if path.read_bytes() != raw:
            raise ValueError(f"Saved source differs: {path}. Keep its code or use a new experiment ID.")
        return
    path.write_bytes(raw)


def make_bundle(folder, output_path, archive_root):
    folder, output_path, archive_root = Path(folder), Path(output_path), Path(archive_root)
    temporary = output_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(archive_root))
    os.replace(temporary, output_path)
    return output_path



def split_bundle(zip_path, part_bytes=256 * 2**20, keep_zip=False):
    """Create checked .bin parts for Kaggle's output-file download buttons."""
    import shutil
    zip_path = Path(zip_path)
    if part_bytes <= 0:
        raise ValueError("Part size must be positive")
    zip_hash = sha256_file(zip_path)
    folder = zip_path.parent / (zip_path.stem + "_parts_" + zip_hash[:12])
    folder.mkdir(parents=True, exist_ok=True)
    manifest_path = folder / "manifest.json"
    if manifest_path.exists():
        old = read_json(manifest_path)
        if (old.get("format") == "cs781_split_zip_v1"
                and old["zip_sha256"] == zip_hash and old["part_bytes"] == part_bytes
                and bool(old["parts"])
                and sum(row["size_bytes"] for row in old["parts"]) == zip_path.stat().st_size
                and all((folder / row["name"]).is_file()
                        and (folder / row["name"]).stat().st_size == row["size_bytes"]
                        and sha256_file(folder / row["name"]) == row["sha256"]
                        for row in old["parts"])):
            if not keep_zip:
                zip_path.unlink()
            return manifest_path
    if shutil.disk_usage(folder).free < zip_path.stat().st_size + 512 * 2**20:
        raise RuntimeError("Not enough disk to split the ZIP safely. Preserve the run folder in saved outputs.")
    rows = []
    whole = hashlib.sha256()
    with open(zip_path, "rb") as source:
        number = 1
        while True:
            first = source.read(min(8 * 2**20, part_bytes))
            if not first:
                break
            part = folder / f"backup.part{number:03d}.bin"
            temporary = part.with_suffix(part.suffix + ".tmp")
            digest = hashlib.sha256()
            written = 0
            with open(temporary, "wb") as target:
                chunk = first
                while chunk:
                    target.write(chunk)
                    digest.update(chunk)
                    whole.update(chunk)
                    written += len(chunk)
                    chunk = source.read(min(8 * 2**20, part_bytes - written)) if written < part_bytes else b""
                target.flush()
                os.fsync(target.fileno())
            os.replace(temporary, part)
            rows.append({"name": part.name, "size_bytes": written, "sha256": digest.hexdigest()})
            number += 1
    if whole.hexdigest() != zip_hash:
        raise ValueError("ZIP changed while splitting; keep the ZIP and try again after training stops")
    atomic_json(manifest_path, {"format": "cs781_split_zip_v1", "zip_name": zip_path.name,
        "zip_size_bytes": zip_path.stat().st_size, "zip_sha256": zip_hash,
        "part_bytes": part_bytes, "parts": rows})
    if not keep_zip:
        zip_path.unlink()  # the same backup bytes remain in the checked parts
    return manifest_path


def reassemble_bundle(manifest_path, output_folder):
    """Restore the original ZIP with bounded RAM and verify every part."""
    manifest_path = Path(manifest_path)
    manifest = read_json(manifest_path)
    if manifest.get("format") != "cs781_split_zip_v1":
        raise ValueError("Unknown split-backup format")
    name = manifest["zip_name"]
    if Path(name).name != name:
        raise ValueError("ZIP name must be a plain filename")
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    output = output_folder / name
    if output.exists():
        if output.stat().st_size == manifest["zip_size_bytes"] and sha256_file(output) == manifest["zip_sha256"]:
            return output
        raise ValueError(f"A different file already exists: {output}")
    temporary = output.with_suffix(output.suffix + ".tmp")
    whole = hashlib.sha256()
    total = 0
    try:
        with open(temporary, "wb") as target:
            for row in manifest["parts"]:
                if Path(row["name"]).name != row["name"]:
                    raise ValueError("Part name must be a plain filename")
                part = manifest_path.parent / row["name"]
                if part.stat().st_size != row["size_bytes"]:
                    raise ValueError(f"Part size differs: {part}")
                digest = hashlib.sha256()
                with open(part, "rb") as source:
                    for chunk in iter(lambda: source.read(8 * 2**20), b""):
                        target.write(chunk)
                        whole.update(chunk)
                        digest.update(chunk)
                        total += len(chunk)
                if digest.hexdigest() != row["sha256"]:
                    raise ValueError(f"Part checksum differs: {part}")
            target.flush()
            os.fsync(target.fileno())
        if total != manifest["zip_size_bytes"] or whole.hexdigest() != manifest["zip_sha256"]:
            raise ValueError("Reassembled ZIP checksum differs")
        os.replace(temporary, output)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return output
'''
(SRC_DIR / 'phase1_utils.py').write_bytes(_module_text.encode('utf-8'))


7327

## 4. Input and restore utilities

Resolve extracted inputs and restore a compatible run into the working directory when a backup is supplied.


In [3]:
_module_text = r'''"""Restore extracted Kaggle input folders, without assuming the ZIP survives."""
import shutil
import zipfile
from pathlib import Path
from phase1_utils import sha256_file


def find_input_folder(folder_name, marker, explicit=None, identical_files=()):
    if explicit:
        folder = Path(explicit)
        if not (folder / marker).is_file():
            raise FileNotFoundError(f"Expected {marker} in {folder}")
        return folder
    matches = sorted({p.parent for p in Path("/kaggle/input").rglob(marker)
                      if p.parent.name == folder_name})
    if not matches:
        return None
    if len(matches) == 1:
        return matches[0]
    if identical_files and all(all((p / name).exists() for name in identical_files) for p in matches):
        fingerprints = [tuple(sha256_file(p / name) for name in identical_files) for p in matches]
        if len(set(fingerprints)) == 1:
            return matches[0]
    raise ValueError(f"Several input copies found. Set the explicit folder path: {matches}")


def restore_folder(destination, folder_name, marker, explicit=None):
    destination = Path(destination)
    if (destination / marker).exists():
        return destination
    source = find_input_folder(folder_name, marker, explicit)
    if source is not None:
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("Restored extracted input folder:", source, flush=True)
    return destination


def restore_zip(zip_path, working_root):
    """Optional path for a ZIP that Kaggle did not extract automatically."""
    working_root = Path(working_root).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            destination = (working_root / member.filename).resolve()
            if not destination.is_relative_to(working_root):
                raise ValueError("Unsafe archive path")
            if destination.exists() and not member.is_dir():
                raise ValueError(f"ZIP would overwrite {destination}; restore into a fresh working folder")
        archive.extractall(working_root)
'''
(SRC_DIR / 'phase1_paths.py').write_bytes(_module_text.encode('utf-8'))


2156

## 5. Byte-BPE implementation

Load the course assignment tokenizer and restore its learned merges from the shared 16K state.


In [4]:
_module_text = r'''"""The assignment byte-BPE algorithm, with checkpoints between merges.

Pair counting, lexicographic tie breaking, non-overlapping replacements and
merge-order encoding are unchanged from the completed Phase-1/A3 tokenizer.
"""
import pickle
import re
import time
from collections import Counter
from pathlib import Path
from phase1_utils import atomic_pickle, sha256_file, text_hash


class A1BPETokenizer:
    def __init__(self):
        specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.vocab = [s.encode("utf-8") for s in specials]
        self.vocab += [bytes([b]) for b in range(256)]
        self.boundary_id = len(self.vocab)
        self.vocab.append(b"")
        self.names = [s.encode("utf-8") for s in specials]
        self.names += [bytes([b]) for b in range(256)]
        self.names.append(b"</w>")
        self.merges = {}
        self.vocab_size = len(self.vocab)

    def fit(self, text, vocab_size, checkpoint_path=None, checkpoint_merges=100,
            stop_after_merges=None):
        if vocab_size < 261:
            raise ValueError("Vocabulary must accommodate bytes and reserved tokens")
        signature = {"text_sha256": text_hash(text), "vocab_size": vocab_size,
                     "algorithm_source_sha256": sha256_file(__file__)}
        checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        elapsed_before = 0.0
        if checkpoint_path is not None and checkpoint_path.exists():
            with open(checkpoint_path, "rb") as file:
                saved = pickle.load(file)
            if saved["signature"] != signature:
                raise ValueError("BPE checkpoint settings or source differ")
            self.__dict__.update(saved["tokenizer_state"])
            words = saved["words"]
            elapsed_before = saved["elapsed_seconds"]
            print("Resuming BPE vocabulary:", self.vocab_size, flush=True)
        else:
            if self.merges:
                raise ValueError("Use a fresh tokenizer or its saved fitting checkpoint")
            words = Counter(tuple(b + 4 for b in piece.encode("utf-8")) + (self.boundary_id,)
                            for piece in re.findall(r"\S+|\s+", text))
        started = time.monotonic()
        last_saved = started
        merges_this_call = 0
        commit_in_progress = False

        def snapshot():
            if checkpoint_path is not None:
                atomic_pickle(checkpoint_path, {"signature": signature,
                    "tokenizer_state": self.__dict__, "words": words,
                    "elapsed_seconds": elapsed_before + time.monotonic() - started})

        try:
            while len(self.vocab) < vocab_size:
                pair_counts = Counter()
                for symbols, frequency in words.items():
                    for pair in zip(symbols, symbols[1:]):
                        pair_counts[pair] += frequency
                if not pair_counts:
                    break
                pair = min(pair_counts, key=lambda p: (-pair_counts[p], self.names[p[0]], self.names[p[1]]))
                if pair_counts[pair] < 2:
                    break
                new_id = len(self.vocab)
                updated = Counter()
                for symbols, frequency in words.items():
                    result = []
                    j = 0
                    while j < len(symbols):
                        if j + 1 < len(symbols) and symbols[j:j + 2] == pair:
                            result.append(new_id)
                            j += 2
                        else:
                            result.append(symbols[j])
                            j += 1
                    updated[tuple(result)] += frequency
                # Commit the whole merge only after every word has been replaced.
                commit_in_progress = True
                self.vocab.append(self.vocab[pair[0]] + self.vocab[pair[1]])
                self.names.append(self.names[pair[0]] + self.names[pair[1]])
                self.merges[pair] = new_id
                self.vocab_size = len(self.vocab)
                words = updated
                commit_in_progress = False
                merges_this_call += 1
                if len(self.merges) % checkpoint_merges == 0 or time.monotonic() - last_saved >= 60:
                    snapshot()
                    print(f"BPE vocabulary {self.vocab_size:,}/{vocab_size:,} | "
                          f"elapsed {(elapsed_before + time.monotonic() - started)/60:.1f} min", flush=True)
                    last_saved = time.monotonic()
                if stop_after_merges is not None and merges_this_call >= stop_after_merges:
                    break
        finally:
            if not commit_in_progress:
                snapshot()
        return self

    def encode(self, text):
        result = []
        for piece in re.findall(r"\S+|\s+", text):
            symbols = [b + 4 for b in piece.encode("utf-8")] + [self.boundary_id]
            while len(symbols) > 1:
                candidates = [(self.merges[(a, b)], (a, b))
                              for a, b in zip(symbols, symbols[1:]) if (a, b) in self.merges]
                if not candidates:
                    break
                new_id, pair = min(candidates)
                merged = []
                j = 0
                while j < len(symbols):
                    if j + 1 < len(symbols) and (symbols[j], symbols[j + 1]) == pair:
                        merged.append(new_id)
                        j += 2
                    else:
                        merged.append(symbols[j])
                        j += 1
                symbols = merged
            result.extend(symbols)
        return result

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")
'''
(SRC_DIR / 'a1_tokenizer.py').write_bytes(_module_text.encode('utf-8'))


5815

## 6. Token caches and source batches

Read the per-source arrays and sample deterministic context blocks while tracking source cursors.


In [5]:
_module_text = r'''"""Stream legal documents, freeze document ownership, and build token caches."""
import json
import os
import re
import time
from functools import lru_cache
from pathlib import Path
import ijson
import numpy as np
from phase1_utils import atomic_json, checked_json, read_json, sha256_file, text_hash

DTYPE = np.dtype("<u2")


def is_validation(digest):
    return int(digest[:16], 16) % 20 == 0


def iter_records(path):
    with open(path, "rb") as file:
        first = file.read(4096).lstrip()[:1]
        file.seek(0)
        if first == b"[":
            records = ijson.items(file, "item")
        elif first == b"{":
            records = (record for _, record in ijson.kvitems(file, ""))
        else:
            raise ValueError("Expected a JSON list or an ID-to-record dictionary")
        for record in records:
            if not isinstance(record, dict) or not isinstance(record.get("text"), str):
                raise ValueError("Every document must contain a string 'text'")
            yield record


def audit_source(path, source, folder, signature, seen,
                 bpe_chars=500_000, val_chars=500_000, generation_examples=25):
    """Exact duplicates belong to the first source in the frozen source order."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    sample_path, manifest_path = folder / f"{source}_samples.json", folder / f"{source}_ownership.jsonl"
    if sample_path.exists():
        saved = read_json(sample_path)
        if saved["signature"] != signature or sha256_file(manifest_path) != saved["ownership_sha256"]:
            raise ValueError("Saved source audit differs")
        with open(manifest_path, encoding="utf-8") as file:
            for line in file:
                digest = json.loads(line)["full_text_sha256"]
                if digest in seen:
                    raise ValueError("Cross-source ownership contains a duplicate")
                seen.add(digest)
        return saved
    temporary = manifest_path.with_suffix(".jsonl.tmp")
    bpe_chunks, val_texts, prompts, truths = [], [], {}, {}
    counts = {"records": 0, "empty": 0, "duplicates": 0, "owned_documents": 0,
              "owned_training_documents": 0, "owned_validation_documents": 0,
              "owned_training_characters": 0, "owned_validation_characters": 0,
              "bpe_characters": 0, "validation_sample_characters": 0}
    last_print = time.monotonic()
    with open(temporary, "w", encoding="utf-8") as manifest:
        for index, record in enumerate(iter_records(path)):
            counts["records"] += 1
            text = record["text"]
            if not text.strip():
                counts["empty"] += 1
                continue
            digest = text_hash(text)
            if digest in seen:
                counts["duplicates"] += 1
                continue
            seen.add(digest)
            validation = is_validation(digest)
            manifest.write(json.dumps({"record_index": index, "full_text_sha256": digest,
                                       "validation": validation, "characters": len(text)}) + "\n")
            counts["owned_documents"] += 1
            key = "validation" if validation else "training"
            counts[f"owned_{key}_documents"] += 1
            counts[f"owned_{key}_characters"] += len(text)
            if not validation and counts["bpe_characters"] < bpe_chars:
                chunk = text[:min(4000, bpe_chars - counts["bpe_characters"])]
                bpe_chunks.append(chunk)
                counts["bpe_characters"] += len(chunk)
            if validation:
                if counts["validation_sample_characters"] < val_chars:
                    chunk = text[:min(20_000, val_chars - counts["validation_sample_characters"])]
                    val_texts.append(chunk)
                    counts["validation_sample_characters"] += len(chunk)
                if len(prompts) < generation_examples:
                    words = text.split(maxsplit=640)
                    if len(words) >= 640:
                        prompt_id = f"{source}_{len(prompts):04d}"
                        prompts[prompt_id] = {"text": " ".join(words[:512])}
                        truths[prompt_id] = {"text": " ".join(words[512:640])}
            if time.monotonic() - last_print >= 30:
                print(source, "| scanned", counts["records"], "| duplicates", counts["duplicates"], flush=True)
                last_print = time.monotonic()
        manifest.flush()
        os.fsync(manifest.fileno())
    os.replace(temporary, manifest_path)
    if counts["bpe_characters"] != bpe_chars or counts["validation_sample_characters"] != val_chars:
        raise ValueError(f"Sample quotas not met for {source}: {counts}")
    if len(prompts) != generation_examples:
        raise ValueError(f"Only {len(prompts)} suitable held-out prompts in {source}")
    saved = {"signature": signature, "statistics": counts, "bpe_chunks": bpe_chunks,
             "validation_texts": val_texts, "prompts": prompts, "truths": truths,
             "ownership_sha256": sha256_file(manifest_path)}
    atomic_json(sample_path, saved)
    return saved


def cached_encoder(tokenizer):
    @lru_cache(maxsize=200_000)
    def piece_ids(piece):
        return tuple(tokenizer.encode(piece))
    def encode(text):
        return [i for piece in re.findall(r"\S+|\s+", text) for i in piece_ids(piece)]
    return encode


def cache_metadata(binary, metadata_path, signature):
    if not Path(metadata_path).exists():
        return None
    metadata = read_json(metadata_path)
    if metadata["signature"] != signature:
        raise ValueError("Token-cache signature differs")
    if Path(binary).stat().st_size != metadata["token_count"] * DTYPE.itemsize:
        raise ValueError("Token-cache size differs")
    if sha256_file(binary) != metadata["sha256"]:
        raise ValueError("Token-cache checksum differs")
    return metadata


def build_source_cache(path, ownership_path, encode, folder, signature, target_tokens):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    binary, metadata_path = folder / "train.bin", folder / "train_metadata.json"
    old = cache_metadata(binary, metadata_path, signature)
    if old is not None:
        return old
    owners = {}
    with open(ownership_path, encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            if not row["validation"]:
                owners[row["record_index"]] = row["full_text_sha256"]
    partial, progress_path = folder / "train.partial.bin", folder / "cache_progress.json"
    state = {"tokens_written": 0, "next_record_index": 0, "documents_encoded": 0,
             "last_document_truncated": False, "complete": False,
             "source_exhausted": False}
    if progress_path.exists():
        progress = read_json(progress_path)
        if progress["signature"] != signature:
            raise ValueError("Partial-cache signature differs")
        state = progress["state"]
    elif partial.exists():
        raise ValueError("Partial cache exists without progress metadata")
    if state["tokens_written"] and not partial.exists() and not binary.exists():
        raise ValueError("The partial cache needed for resumption is missing")
    if not state["complete"]:
        last_print = time.monotonic()
        with open(partial, "r+b" if partial.exists() else "w+b") as output:
            output.seek(state["tokens_written"] * DTYPE.itemsize)
            output.truncate()
            def snapshot():
                output.seek(state["tokens_written"] * DTYPE.itemsize)
                output.truncate()
                output.flush()
                os.fsync(output.fileno())
                atomic_json(progress_path, {"signature": signature, "state": state})
            try:
                for index, record in enumerate(iter_records(path)):
                    if index < state["next_record_index"]:
                        continue
                    if index in owners:
                        if text_hash(record["text"]) != owners[index]:
                            raise ValueError("Document content differs from the frozen audit")
                        ids = encode(record["text"]) + [3]
                        keep = ids[:target_tokens - state["tokens_written"]]
                        np.asarray(keep, dtype=DTYPE).tofile(output)
                        updated = dict(state)
                        updated["tokens_written"] += len(keep)
                        updated["documents_encoded"] += 1
                        updated["last_document_truncated"] = len(keep) < len(ids)
                        updated["next_record_index"] = index + 1
                        state = updated
                    else:
                        state["next_record_index"] = index + 1
                    if time.monotonic() - last_print >= 30:
                        snapshot()
                        print(signature["source"], "| cached", state["tokens_written"], "/", target_tokens, flush=True)
                        last_print = time.monotonic()
                    if state["tokens_written"] >= target_tokens:
                        state["complete"] = True
                        break
                else:
                    state["source_exhausted"] = True
                    state["complete"] = True
            finally:
                snapshot()
    source = partial if partial.exists() else binary
    if not source.exists() or state["tokens_written"] == 0:
        raise ValueError("No training tokens in this source")
    checksum = sha256_file(source)
    if partial.exists():
        os.replace(partial, binary)
    metadata = {"signature": signature, "dtype": DTYPE.str,
                "token_count": state["tokens_written"], "sha256": checksum,
                "requested_token_count": target_tokens,
                "source_exhausted": state["source_exhausted"], "statistics": state}
    atomic_json(metadata_path, metadata)
    return metadata


def build_validation_cache(texts, encode, folder, signature):
    folder = Path(folder)
    binary, metadata_path = folder / "validation.bin", folder / "validation_metadata.json"
    old = cache_metadata(binary, metadata_path, signature)
    if old is not None:
        return old
    temporary = binary.with_suffix(".bin.tmp")
    count = 0
    with open(temporary, "wb") as output:
        for text in texts:
            ids = encode(text) + [3]
            np.asarray(ids, dtype=DTYPE).tofile(output)
            count += len(ids)
    os.replace(temporary, binary)
    metadata = {"signature": signature, "dtype": DTYPE.str,
                "token_count": count, "sha256": sha256_file(binary), "documents": len(texts)}
    atomic_json(metadata_path, metadata)
    return metadata


def block_order(n_blocks, seed, source_number, epoch):
    return np.random.default_rng(seed + 100_000 * source_number + epoch).permutation(n_blocks)


def initial_cursors(sources):
    return {source: {"epoch": 0, "cursor": 0} for source in sources}


def mixed_batch(caches, cursors, source_order, source_offset, batch_size, block_size, seed, orders=None):
    xs, ys, counts = [], [], {source: 0 for source in source_order}
    for j in range(batch_size):
        source_number = (source_offset + j) % len(source_order)
        source = source_order[source_number]
        tokens = caches[source]
        n_blocks = (len(tokens) - 1) // block_size
        state = cursors[source]
        if state["cursor"] == n_blocks:
            state["epoch"] += 1
            state["cursor"] = 0
            if orders is not None:
                orders[source] = block_order(n_blocks, seed, source_number, state["epoch"])
        order = (block_order(n_blocks, seed, source_number, state["epoch"])
                 if orders is None else orders[source])
        index = int(order[state["cursor"]])
        state["cursor"] += 1
        start = index * block_size
        xs.append(np.asarray(tokens[start:start + block_size], dtype=np.int64))
        ys.append(np.asarray(tokens[start + 1:start + block_size + 1], dtype=np.int64))
        counts[source] += block_size
    return np.stack(xs), np.stack(ys), (source_offset + batch_size) % len(source_order), counts
'''
(SRC_DIR / 'phase1_data.py').write_bytes(_module_text.encode('utf-8'))


12279

## 7. Model implementation

Write the matching architecture definition. Source hashes are checked against the training backup before weights are loaded.


In [6]:
_module_text = r'''"""Efficient causal LM: assignment GQA with actual sliding-window attention.

Course Module 2.3 pages 155-156, 167-170 and 180-185.
A3 Q2 supplies the KV-head sharing algorithm. SDPA Boolean True means allowed.
Training uses next-token targets. KV caches are inference-only.
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "efficient"
    vocab_size: int = 16000
    d_model: int = 1024
    n_heads: int = 16
    n_layers: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4864
    max_seq_len: int = 2560
    dropout: float = 0.1
    window_size: int = 512
    query_chunk_size: int = 128

    def __post_init__(self):
        if self.architecture != "efficient":
            raise ValueError("This notebook implements the efficient architecture only")
        assert self.d_model % self.n_heads == 0 and self.n_heads % self.n_kv_heads == 0
        assert min(self.vocab_size - 4, self.n_layers, self.d_ff, self.max_seq_len,
                   self.window_size, self.query_chunk_size) > 0
        assert 0 <= self.dropout < 1


def configuration(architecture="efficient"):
    return ModelConfig(architecture=architecture)



def positions(max_len, width):
    p = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, width, 2).float() * (-math.log(10000.0) / width))
    result = torch.zeros(max_len, width)
    result[:, 0::2] = torch.sin(p * frequency)
    result[:, 1::2] = torch.cos(p * frequency)
    return result


class Attention(nn.Module):
    def __init__(self, cfg, kv_heads=None):
        super().__init__()
        self.heads = cfg.n_heads
        self.kv_heads = cfg.n_heads if kv_heads is None else kv_heads
        assert cfg.d_model % self.heads == 0 and self.heads % self.kv_heads == 0
        self.head_dim = cfg.d_model // self.heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def query(self, x):
        b, t, _ = x.shape
        return self.q_proj(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)

    def keys_values(self, x):
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        return k, v

    def attend(self, q, k, v, allowed=None, causal=False):
        # A3 Q2: repeat each KV head for its group of query heads.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k, v = k.repeat_interleave(repeats, dim=1), v.repeat_interleave(repeats, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0, scale=1 / math.sqrt(self.head_dim))
        b, _, t, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(b, t, -1)
        return self.resid_dropout(self.out_proj(out))

    def forward(self, x, memory=None, allowed=None, causal=False):
        k, v = self.keys_values(x if memory is None else memory)
        return self.attend(self.query(x), k, v, allowed, causal)

    def local(self, x, window, chunk, lengths=None):
        # Query tiles see only nearby keys. No full T-by-T score matrix.
        q = self.query(x)
        k, v = self.keys_values(x)
        t = x.shape[1]
        pieces = []
        for start in range(0, t, chunk):
            stop = min(t, start + chunk)
            first = max(0, start - window + 1)
            qi = torch.arange(start, stop, device=x.device)
            ki = torch.arange(first, stop, device=x.device)
            allowed = (ki[None, :] <= qi[:, None]) & (ki[None, :] > qi[:, None] - window)
            allowed = allowed[None, None, :, :]
            if lengths is not None:
                allowed = allowed & (ki[None, :] < lengths[:, None])[:, None, None, :]
            pieces.append(self.attend(q[:, :, start:stop], k[:, :, first:stop], v[:, :, first:stop], allowed))
        return torch.cat(pieces, dim=1)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
                                 nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)


class EfficientBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg, cfg.n_kv_heads), FeedForward(cfg)
        self.window, self.chunk = cfg.window_size, cfg.query_chunk_size

    def forward(self, x, lengths=None):
        x = x + self.attn.local(self.ln1(x), self.window, self.chunk, lengths)
        return x + self.ffn(self.ln2(x))


class BaseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", positions(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def initialize(self):
        for p in self.parameters():
            if p.ndim > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def embed(self, ids, offset=None):
        t = ids.shape[1]
        p = torch.arange(t, device=ids.device)[None, :]
        if offset is not None:
            p = p + offset[:, None]
        x = self.tok_emb(ids) * math.sqrt(self.cfg.d_model) + self.pos_emb[p]
        return x


class EfficientLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.blocks = nn.ModuleList([EfficientBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, ids, lengths=None):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Input exceeds context budget")
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, lengths)
        return self.head(self.final_norm(x))

    def start_generation(self, prompt, lengths):
        x = self.embed(prompt)
        layers = []
        keep = self.cfg.window_size - 1
        indices = lengths[:, None] - keep + torch.arange(keep, device=prompt.device)[None, :]
        gather = indices.clamp_min(0)[:, None, :, None]
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            shape = (prompt.shape[0], k.shape[1], keep, k.shape[-1])
            layers.append({"k": torch.gather(k, 2, gather.expand(shape)),
                           "v": torch.gather(v, 2, gather.expand(shape))})
            x = x + block.attn.local(h, block.window, block.chunk, lengths)
            x = x + block.ffn(block.ln2(x))
        last = self.final_norm(x)[torch.arange(x.shape[0], device=x.device), lengths - 1]
        cache = {"layers": layers, "lengths": lengths, "next_positions": lengths.clone(),
                 "key_positions": indices}
        return self.head(last), cache

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"])
        key_positions = torch.cat([cache["key_positions"], cache["next_positions"][:, None]], dim=1)
        allowed = (key_positions >= 0)[:, None, None, :]
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            k, v = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
            layer["k"], layer["v"] = k[:, :, 1:], v[:, :, 1:]
        cache["key_positions"] = key_positions[:, 1:]
        cache["next_positions"] = cache["next_positions"] + 1
        return self.head(self.final_norm(x))[:, -1], cache


def select_cache(cache, indices):
    """Remove finished rows only when the active batch shrinks."""
    if isinstance(cache, torch.Tensor):
        return cache.index_select(0, indices)
    if isinstance(cache, dict):
        return {k: select_cache(v, indices) for k, v in cache.items()}
    if isinstance(cache, list):
        return [select_cache(v, indices) for v in cache]
    return cache


class ModelWithLoss(nn.Module):
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm, self.use_amp = lm, use_amp

    def forward(self, x, y, prefixes):
        # DataParallel scatters this per-example tensor along with x and y.
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits, labels = logits_and_labels(self.lm, x, y, 0)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), labels.reshape(-1))
        return loss.unsqueeze(0)


def build_model(cfg):
    return EfficientLM(cfg)


def logits_and_labels(lm, x, y, prefix_length=0):
    # y is the token immediately after each input position in x.
    return lm(x), y
'''
(SRC_DIR / 'architecture_models.py').write_bytes(_module_text.encode('utf-8'))


9705

## 8. Training and checkpoint source

Write the training helpers used to interpret the checkpoint and recover its saved run settings.


In [7]:
_module_text = r'''"""Shared training: input-token budgets, continuation-target counts, full resume."""
import copy
import gc
import math
import os
import random
import signal
import shutil
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from phase1_utils import atomic_json, checked_json, read_json, sha256_file, make_bundle
from phase1_data import DTYPE, block_order, initial_cursors, mixed_batch
from architecture_models import ModelConfig, build_model, ModelWithLoss


def cosine_multiplier(step, total_steps, warmup_steps, minimum_fraction):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
    return minimum_fraction + (1 - minimum_fraction) * 0.5 * (1 + math.cos(math.pi * progress))


def atomic_torch_save(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        torch.save(value, file)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def load_caches(prep_dir):
    prep_dir = Path(prep_dir)
    manifest = read_json(prep_dir / "cache_manifest.json")
    training, validation = {}, {}
    for source in manifest["source_order"]:
        training[source] = np.memmap(prep_dir / "caches" / source / "train.bin", dtype=DTYPE, mode="r")
        validation[source] = np.memmap(prep_dir / "caches" / source / "validation.bin", dtype=DTYPE, mode="r")
    return manifest, training, validation


def gradient_norm(parameters):
    """Measure unscaled gradients without changing them, including nonfinite ones."""
    norms = [torch.linalg.vector_norm(p.grad.detach(), dtype=torch.float32)
             for p in parameters if p.grad is not None]
    return torch.linalg.vector_norm(torch.stack(norms))


class NonfiniteTrainingLoss(RuntimeError):
    pass


@torch.no_grad()
def evaluate_sources(loss_model, caches, block_size, batch_size, device):
    loss_model.eval()
    measurements = {}
    conditional = loss_model.lm.cfg.architecture != "efficient"
    for source, tokens in caches.items():
        loss_sum, targets, inputs = 0.0, 0, 0
        for start in range(0, len(tokens) - 1, block_size * batch_size):
            lengths = [min(block_size, len(tokens) - 1 - offset)
                       for offset in range(start, min(len(tokens) - 1, start + block_size * batch_size), block_size)]
            # Full blocks can be batched; the last short block is measured alone.
            full_lengths = [length for length in lengths if length == block_size]
            if full_lengths:
                offsets = [start + j * block_size for j in range(len(full_lengths))]
                x = np.stack([np.asarray(tokens[o:o + block_size], dtype=np.int64) for o in offsets])
                y = np.stack([np.asarray(tokens[o + 1:o + block_size + 1], dtype=np.int64) for o in offsets])
                xt, yt = torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)
                p = int(block_size * 0.6) if conditional else 0
                prefix = torch.full((len(offsets),), p, device=device, dtype=torch.long)
                count = len(offsets) * (block_size - p)
                loss_sum += float(loss_model(xt, yt, prefix).mean().item()) * count
                targets += count
                inputs += x.size
            if lengths and lengths[-1] < block_size:
                offset = start + (len(lengths) - 1) * block_size
                xt = torch.from_numpy(np.asarray(tokens[offset:-1], dtype=np.int64)[None]).to(device)
                yt = torch.from_numpy(np.asarray(tokens[offset + 1:], dtype=np.int64)[None]).to(device)
                if conditional and xt.shape[1] < 2:
                    continue
                p = max(1, int(xt.shape[1] * 0.6)) if conditional else 0
                prefix = torch.full((1,), p, device=device, dtype=torch.long)
                count = xt.shape[1] - p
                loss_sum += float(loss_model(xt, yt, prefix).mean().item()) * count
                targets += count
                inputs += xt.numel()
        if targets == 0 or not math.isfinite(loss_sum):
            raise FloatingPointError("Invalid validation loss or target count")
        nll = loss_sum / targets
        measurements[source] = {"nll": nll, "perplexity": math.exp(nll), "targets": targets,
                                "input_positions": inputs}
    macro_nll = sum(row["nll"] for row in measurements.values()) / len(measurements)
    token_nll = sum(row["nll"] * row["targets"] for row in measurements.values()) / sum(row["targets"] for row in measurements.values())
    return {"validation_nll": macro_nll, "validation_perplexity": math.exp(macro_nll),
            "validation_token_nll": token_nll, "sources": measurements}


def train_run(prep_dir, run_dir, model_config, settings, stop_after_tokens,
              max_run_hours=10.0, cpu_test=False):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    if not cpu_test and (not torch.cuda.is_available() or torch.cuda.device_count() < 2):
        raise RuntimeError("Select T4 x2 for full training")
    device = torch.device("cpu" if cpu_test else "cuda:0")
    manifest, train_caches, val_caches = load_caches(prep_dir)
    cfg = ModelConfig(**model_config)
    sources = manifest["source_order"]
    batch_size = settings["global_batch_size"]
    accumulation = settings["gradient_accumulation_steps"]
    sequences_per_update = batch_size * accumulation
    assert batch_size % 2 == 0 and sequences_per_update % len(sources) == 0
    assert 0 < settings["warmup_fraction"] < 1
    if cfg.architecture != "efficient":
        assert all(0 < p < cfg.max_seq_len for p in settings["prefix_lengths"])
    tokens_per_update = sequences_per_update * cfg.max_seq_len
    total_steps = math.ceil(settings["planned_input_tokens"] / tokens_per_update)
    warmup_steps = max(1, int(total_steps * settings["warmup_fraction"]))
    minimum_fraction = settings["minimum_learning_rate"] / settings["learning_rate"]
    assert 0 < minimum_fraction < 1
    gradient_alarm = settings.get("gradient_norm_alarm", 50.0)
    validation_margin = settings.get("validation_regression_margin", 0.3)
    validation_patience = settings.get("validation_regression_patience", 2)
    overflow_patience = settings.get("maximum_consecutive_overflows", 3)
    assert gradient_alarm > settings["gradient_clip_norm"]
    assert validation_margin > 0 and validation_patience > 0 and overflow_patience > 0
    assert all((len(tokens) - 1) // cfg.max_seq_len > 0 for tokens in train_caches.values())
    protocol = {"model_config": model_config, "settings": settings,
                "cache_manifest": manifest, "tokens_per_optimizer_update": tokens_per_update,
                "planned_optimizer_updates": total_steps, "warmup_optimizer_updates": warmup_steps,
                "model_source_sha256": sha256_file(Path(__file__).with_name("architecture_models.py")),
                "data_source_sha256": sha256_file(Path(__file__).with_name("phase1_data.py")),
                "trainer_source_sha256": sha256_file(__file__),
                "selection_metric": "Equal-source validation NLL within this architecture/objective",
                "validation_rule": "Conditional models score suffix after fixed 60% prefix; efficient scores shifted next tokens",
                "budget_unit": "input positions in successful optimizer updates"}
    checked_json(run_dir / "protocol.json", protocol)
    seed = settings["seed"]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if not cpu_test:
        torch.cuda.manual_seed_all(seed)
    lm = build_model(cfg)
    n_parameters = sum(p.numel() for p in lm.parameters())
    if not cpu_test:
        assert 50_000_000 <= n_parameters <= 300_000_000
    wrapped = ModelWithLoss(lm, use_amp=not cpu_test).to(device)
    model = wrapped if cpu_test else nn.DataParallel(wrapped, device_ids=[0, 1])
    optimizer = torch.optim.Adam(model.parameters(), lr=settings["learning_rate"],
                                 betas=tuple(settings["adam_betas"]), eps=settings["adam_eps"])
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,
        lambda step: cosine_multiplier(step, total_steps, warmup_steps, minimum_fraction))
    scaler = torch.amp.GradScaler("cuda", enabled=not cpu_test,
        init_scale=settings.get("amp_initial_scale", 16384.0))
    state = {"optimizer_updates": 0, "attempted_optimizer_updates": 0,
             "input_tokens_processed": 0, "input_tokens_in_applied_updates": 0, "skipped_optimizer_updates": 0,
             "cursors": initial_cursors(sources), "source_offset": 0,
             "processed_source_inputs": {source: 0 for source in sources},
             "applied_source_inputs": {source: 0 for source in sources},
             "targets_processed": 0, "targets_in_applied_updates": 0,
             "processed_source_targets": {source: 0 for source in sources},
             "applied_source_targets": {source: 0 for source in sources},
             "last_validation_tokens": 0, "last_checkpoint_tokens": 0,
             "best_validation_nll": None, "window_loss_sum": 0.0, "window_targets": 0,
             "window_max_gradient_norm": 0.0, "consecutive_overflows": 0,
             "rejected_gradient_updates": 0, "rejected_forward_updates": 0,
             "validation_regression_streak": 0}
    history = []
    last_path = run_dir / "last.pt"
    resumed_from = None
    if last_path.exists():
        checkpoint = torch.load(last_path, map_location="cpu", weights_only=False)
        if checkpoint["protocol"] != protocol:
            raise ValueError("Checkpoint protocol differs")
        lm.load_state_dict(checkpoint["model_state"], strict=True)
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        scaler.load_state_dict(checkpoint["scaler_state"])
        state, history = checkpoint["training_state"], checkpoint["history"]
        random.setstate(checkpoint["rng_state"]["python"])
        np.random.set_state(checkpoint["rng_state"]["numpy"])
        torch.set_rng_state(checkpoint["rng_state"]["torch_cpu"])
        if not cpu_test:
            torch.cuda.set_rng_state_all(checkpoint["rng_state"]["torch_cuda"])
        resumed_from = str(last_path)
        del checkpoint
    orders = {source: block_order((len(train_caches[source]) - 1) // cfg.max_seq_len,
                                 seed, j, state["cursors"][source]["epoch"])
              for j, source in enumerate(sources)}
    initial_processed, initial_applied = state["input_tokens_processed"], state["input_tokens_in_applied_updates"]
    stop_requested = False
    def request_pause(signum, frame):
        nonlocal stop_requested
        stop_requested = True
        print("Pause requested; finishing this optimizer attempt and saving.", flush=True)
    previous_handler = signal.getsignal(signal.SIGINT)
    signal.signal(signal.SIGINT, request_pause)
    started = time.monotonic()
    last_saved_time = started
    last_backup_time = started
    last_backup_tokens = initial_applied
    safety_reason = None

    def rng_state():
        return {"python": random.getstate(), "numpy": np.random.get_state(),
                "torch_cpu": torch.get_rng_state(),
                "torch_cuda": [] if cpu_test else torch.cuda.get_rng_state_all()}

    def restore_rng(rng):
        random.setstate(rng["python"])
        np.random.set_state(rng["numpy"])
        torch.set_rng_state(rng["torch_cpu"])
        if not cpu_test:
            torch.cuda.set_rng_state_all(rng["torch_cuda"])

    def checkpoint_payload():
        return {"protocol": protocol, "model_state": lm.state_dict(),
            "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(), "training_state": copy.deepcopy(state),
            "history": history, "rng_state": rng_state()}

    def save_checkpoint():
        nonlocal last_saved_time
        state["last_checkpoint_tokens"] = state["input_tokens_in_applied_updates"]
        optimizer.zero_grad(set_to_none=True)
        atomic_torch_save(last_path, checkpoint_payload())
        atomic_json(run_dir / "history.json", history)
        last_saved_time = time.monotonic()
        print("Saved checkpoint at", state["input_tokens_in_applied_updates"], "input positions", flush=True)

    def save_auto_backup(force=False):
        nonlocal last_backup_time, last_backup_tokens
        if not settings.get("automatic_backups", not cpu_test):
            return
        if not force and (state["input_tokens_in_applied_updates"] - last_backup_tokens <
                          settings.get("backup_every_tokens", 50_000_000)
                          and time.monotonic() - last_backup_time <
                          settings.get("backup_every_seconds", 45 * 60)):
            return
        backup_path = run_dir.parents[2] / f"{run_dir.name}_backup.zip"
        required = sum(p.stat().st_size for p in run_dir.rglob("*") if p.is_file())
        if shutil.disk_usage(backup_path.parent).free < required + 512 * 2**20:
            print("ZIP snapshot skipped: insufficient free disk. Preserve the run folder in saved outputs.", flush=True)
            return
        atomic_json(run_dir / "backup_info.json", {
            "checkpoint_input_tokens": state["input_tokens_in_applied_updates"],
            "best_validation_nll": state["best_validation_nll"],
            "zip_path": str(backup_path),
            "note": "A local ZIP still needs downloading or preservation in saved outputs."})
        try:
            make_bundle(run_dir, backup_path, run_dir.parents[2])
        except OSError as error:
            print("ZIP snapshot failed; checkpoint files remain:", str(error), flush=True)
            return
        last_backup_time = time.monotonic()
        last_backup_tokens = state["input_tokens_in_applied_updates"]
        print("Automatic backup ready:", backup_path, "| input positions:", last_backup_tokens,
              "| download it or preserve notebook outputs", flush=True)

    def record_validation():
        nonlocal safety_reason
        measured = evaluate_sources(wrapped, val_caches, cfg.max_seq_len,
                                    settings["validation_batch_size"], device)
        row = {"type": "validation", "optimizer_updates": state["optimizer_updates"],
               "input_tokens_in_applied_updates": state["input_tokens_in_applied_updates"],
               "targets_in_applied_updates": state["targets_in_applied_updates"], **measured}
        history.append(row)
        state["last_validation_tokens"] = state["input_tokens_in_applied_updates"]
        if state["best_validation_nll"] is None or measured["validation_nll"] < state["best_validation_nll"]:
            state["best_validation_nll"] = measured["validation_nll"]
            state["validation_regression_streak"] = 0
            atomic_torch_save(run_dir / "best_model.pt", {"model_config": model_config,
                "model_state": lm.state_dict(), "tokenizer_signature": manifest["tokenizer_signature"],
                "validation_result": row})
            # Full state from this same moment enables an exact rollback later.
            atomic_torch_save(run_dir / "best_training.pt", checkpoint_payload())
        elif measured["validation_nll"] > state["best_validation_nll"] + validation_margin:
            state["validation_regression_streak"] += 1
        else:
            state["validation_regression_streak"] = 0
        if state["validation_regression_streak"] >= validation_patience:
            safety_reason = "validation_regression_alarm"
        atomic_json(run_dir / "history.json", history)
        print("Validation", state["input_tokens_in_applied_updates"], "| macro NLL", measured["validation_nll"],
              "| source NLLs", {k: round(v["nll"], 4) for k, v in measured["sources"].items()}, flush=True)
        model.train()

    run_limit = min(stop_after_tokens, settings["planned_input_tokens"])
    exit_reason = "requested_token_chunk_complete"
    print("Architecture:", cfg.architecture, "| parameters:", n_parameters, "| input positions/update:", tokens_per_update,
          "| planned updates:", total_steps, "| warmup:", warmup_steps, flush=True)
    try:
        if not history:
            record_validation()
            save_checkpoint()
        while state["input_tokens_in_applied_updates"] < run_limit:
            if stop_requested or time.monotonic() - started >= max_run_hours * 3600:
                exit_reason = "manual_pause" if stop_requested else "time_cap"
                break
            model.train()
            optimizer.zero_grad(set_to_none=True)
            attempt_state, attempt_rng = copy.deepcopy(state), rng_state()
            update_loss = 0.0
            update_counts = {source: 0 for source in sources}
            prefix_length = random.choice(settings["prefix_lengths"]) if cfg.architecture != "efficient" else 0
            targets_per_sequence = cfg.max_seq_len - prefix_length
            update_targets = sequences_per_update * targets_per_sequence
            used_lr = optimizer.param_groups[0]["lr"]
            for _ in range(accumulation):
                xa, ya, state["source_offset"], counts = mixed_batch(train_caches, state["cursors"],
                    sources, state["source_offset"], batch_size, cfg.max_seq_len, seed, orders)
                x, y = torch.from_numpy(xa).to(device), torch.from_numpy(ya).to(device)
                prefixes = torch.full((batch_size,), prefix_length, device=device, dtype=torch.long)
                loss = model(x, y, prefixes).mean()
                if not torch.isfinite(loss).item():
                    raise NonfiniteTrainingLoss("Nonfinite forward loss; rejected partial optimizer attempt")
                update_loss += float(loss.item()) / accumulation
                scaler.scale(loss / accumulation).backward()
                state["input_tokens_processed"] += y.numel()
                state["targets_processed"] += batch_size * targets_per_sequence
                for source in sources:
                    state["processed_source_inputs"][source] += counts[source]
                    state["processed_source_targets"][source] += (counts[source] // cfg.max_seq_len) * targets_per_sequence
                    update_counts[source] += counts[source]
            assert set(update_counts.values()) == {tokens_per_update // len(sources)}
            scaler.unscale_(optimizer)
            norm = gradient_norm(model.parameters())
            previous_scale = scaler.get_scale()
            state["attempted_optimizer_updates"] += 1
            if not torch.isfinite(norm).item():
                # Do not clip or apply nonfinite gradients. Explicitly reduce AMP scale.
                scaler.update(new_scale=previous_scale * 0.5)
                skipped = True
                state["consecutive_overflows"] += 1
                if state["consecutive_overflows"] >= overflow_patience:
                    safety_reason = "consecutive_gradient_overflows"
            elif norm.item() > gradient_alarm:
                # A finite but extreme gradient is an alarm, not a useful clipped update.
                scaler.update(new_scale=previous_scale)
                skipped = True
                state["rejected_gradient_updates"] += 1
                safety_reason = "gradient_norm_alarm"
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), settings["gradient_clip_norm"],
                                              error_if_nonfinite=True)
                scaler.step(optimizer)
                scaler.update()
                skipped = scaler.get_scale() < previous_scale
                state["consecutive_overflows"] = state["consecutive_overflows"] + 1 if skipped else 0
            if skipped:
                if safety_reason != "gradient_norm_alarm":
                    state["skipped_optimizer_updates"] += 1
            else:
                state["optimizer_updates"] += 1
                state["input_tokens_in_applied_updates"] += tokens_per_update
                state["targets_in_applied_updates"] += update_targets
                for source in sources:
                    state["applied_source_inputs"][source] += update_counts[source]
                    state["applied_source_targets"][source] += (update_counts[source] // cfg.max_seq_len) * targets_per_sequence
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            state["window_loss_sum"] += update_loss * update_targets
            state["window_targets"] += update_targets
            if torch.isfinite(norm).item():
                state["window_max_gradient_norm"] = max(state["window_max_gradient_norm"], float(norm.item()))
            if safety_reason:
                incident = {"type": "safety_stop", "reason": safety_reason,
                    "input_tokens_in_applied_updates": state["input_tokens_in_applied_updates"],
                    "attempted_optimizer_updates": state["attempted_optimizer_updates"],
                    "gradient_norm_before_clipping": float(norm.item()) if torch.isfinite(norm).item() else None,
                    "gradient_scale": scaler.get_scale(), "prefix_length": prefix_length,
                    "update_nll": update_loss}
                history.append(incident)
                atomic_json(run_dir / "safety_stop.json", incident)
                exit_reason = safety_reason
                print("SAFETY PAUSE:", incident, flush=True)
                break
            if not skipped and state["optimizer_updates"] % settings["log_every_updates"] == 0:
                row = {"type": "training", "optimizer_updates": state["optimizer_updates"],
                    "input_tokens_in_applied_updates": state["input_tokens_in_applied_updates"],
                    "targets_in_applied_updates": state["targets_in_applied_updates"],
                    "prefix_length_this_update": prefix_length,
                    "train_nll_window": state["window_loss_sum"] / state["window_targets"],
                    "learning_rate_used": used_lr, "gradient_scale": scaler.get_scale(),
                    "gradient_norm_before_clipping": float(norm.item()) if torch.isfinite(norm).item() else None,
                    "maximum_gradient_norm_in_window": state["window_max_gradient_norm"],
                    "skipped_optimizer_updates": state["skipped_optimizer_updates"],
                    "elapsed_seconds_this_invocation": time.monotonic() - started}
                history.append(row)
                print("Update", row["optimizer_updates"], "| input positions", row["input_tokens_in_applied_updates"],
                      "| train NLL", round(row["train_nll_window"], 4), "| LR", used_lr,
                      "| max grad", round(row["maximum_gradient_norm_in_window"], 3),
                      "| AMP scale", row["gradient_scale"],
                      "| skipped", row["skipped_optimizer_updates"], flush=True)
                state["window_loss_sum"], state["window_targets"] = 0.0, 0
                state["window_max_gradient_norm"] = 0.0
            if state["input_tokens_in_applied_updates"] - state["last_validation_tokens"] >= settings["validate_every_tokens"]:
                record_validation()
                if safety_reason:
                    exit_reason = safety_reason
                    incident = {"type": "safety_stop", "reason": safety_reason,
                        "input_tokens_in_applied_updates": state["input_tokens_in_applied_updates"],
                        "validation_nll": history[-1]["validation_nll"],
                        "best_validation_nll": state["best_validation_nll"]}
                    history.append(incident)
                    atomic_json(run_dir / "safety_stop.json", incident)
                    print("SAFETY PAUSE:", incident, flush=True)
                    break
            if (state["input_tokens_in_applied_updates"] - state["last_checkpoint_tokens"] >= settings["checkpoint_every_tokens"]
                    or time.monotonic() - last_saved_time >= settings["checkpoint_every_seconds"]):
                save_checkpoint()
            if (state["input_tokens_in_applied_updates"] - last_backup_tokens >=
                    settings.get("backup_every_tokens", 50_000_000)
                    or time.monotonic() - last_backup_time >= settings.get("backup_every_seconds", 45 * 60)):
                if settings.get("automatic_backups", not cpu_test):
                    save_checkpoint()
                    save_auto_backup()
            if state["skipped_optimizer_updates"] >= settings["maximum_skipped_updates"]:
                exit_reason = "too_many_skipped_updates"
                break
        if state["last_validation_tokens"] != state["input_tokens_in_applied_updates"]:
            record_validation()
        save_checkpoint()
    except NonfiniteTrainingLoss as error:
        # No optimizer step has occurred in this partial attempt. Rewind data/RNG.
        state = attempt_state
        restore_rng(attempt_rng)
        state["attempted_optimizer_updates"] += 1
        state["rejected_forward_updates"] += 1
        optimizer.zero_grad(set_to_none=True)
        exit_reason = "nonfinite_training_loss"
        incident = {"type": "safety_stop", "reason": exit_reason,
            "input_tokens_in_applied_updates": state["input_tokens_in_applied_updates"], "message": str(error)}
        history.append(incident)
        atomic_json(run_dir / "safety_stop.json", incident)
        print("SAFETY PAUSE:", incident, flush=True)
        save_checkpoint()
    finally:
        signal.signal(signal.SIGINT, previous_handler)
    elapsed = time.monotonic() - started
    summary = {"exit_reason": exit_reason, "resumed_from": resumed_from,
        "architecture": cfg.architecture, "parameters": n_parameters, "training_state": state,
        "requested_chunk_end_input_tokens": stop_after_tokens,
        "training_loop_elapsed_seconds_this_invocation": elapsed,
        "new_input_tokens_in_applied_updates": state["input_tokens_in_applied_updates"] - initial_applied,
        "new_input_tokens_processed": state["input_tokens_processed"] - initial_processed,
        "input_tokens_per_second": (state["input_tokens_processed"] - initial_processed) / max(elapsed, 1e-8),
        "torch_version": str(torch.__version__), "cuda_version": torch.version.cuda}
    atomic_json(run_dir / "run_summary.json", summary)
    summaries_path = run_dir / "training_invocations.json"
    summaries = read_json(summaries_path) if summaries_path.exists() else []
    summaries.append(summary)
    atomic_json(summaries_path, summaries)
    save_auto_backup(force=True)
    del model, wrapped, lm, optimizer, scheduler, scaler
    gc.collect()
    if not cpu_test:
        torch.cuda.empty_cache()
    return summary
'''
(SRC_DIR / 'architecture_training.py').write_bytes(_module_text.encode('utf-8'))


27099

## 9. Numerical probes

Load the checks that compare cached decoding with a full forward pass and verify causality and padding behavior.


In [8]:
_module_text = r'''"""Discarded fitting/throughput probes and cached-versus-full forward checks."""
import gc
import time
import numpy as np
import torch
import torch.nn as nn
from architecture_models import ModelConfig, build_model, ModelWithLoss, logits_and_labels
from phase1_data import block_order, initial_cursors, mixed_batch


def require_two_gpus():
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("Select T4 x2 for the training probes")


def memory():
    return [{"gpu": j, "total_gib": torch.cuda.get_device_properties(j).total_memory / 2**30,
             "peak_allocated_gib": torch.cuda.max_memory_allocated(j) / 2**30,
             "peak_reserved_gib": torch.cuda.max_memory_reserved(j) / 2**30}
            for j in range(2)]


@torch.no_grad()
def numerical_check(lm, tolerance=1e-3):
    """Check padding, causality, local masking and rolling KV reuse in FP32."""
    was_training = lm.training
    lm.eval()
    device = next(lm.parameters()).device
    max_cache_error, max_padding_error = 0.0, 0.0
    cache_shape_ok = True
    # Both a short prompt and a prompt longer than the attention window.
    widths = [13, min(lm.cfg.window_size + 3, lm.cfg.max_seq_len - 4)]
    for width in sorted(set(widths)):
        prompt = torch.randint(4, lm.cfg.vocab_size, (2, width), device=device)
        lengths = torch.tensor([max(1, width - 4), width], device=device)
        next_ids = torch.randint(4, lm.cfg.vocab_size, (2, 3), device=device)
        logits, cache = lm.start_generation(prompt, lengths)
        full = lm(prompt, lengths)
        for j, length in enumerate(lengths.tolist()):
            one = lm(prompt[j:j+1, :length])
            max_padding_error = max(max_padding_error, float((one[0] - full[j, :length]).abs().max().item()))
        for step in range(3):
            if step:
                logits, cache = lm.generation_step(next_ids[:, step - 1:step], cache)
            for j, length in enumerate(lengths.tolist()):
                known = torch.cat([prompt[j:j+1, :length], next_ids[j:j+1, :step]], dim=1)
                one = lm(known)[:, -1]
                max_cache_error = max(max_cache_error, float((one[0] - logits[j]).abs().max().item()))
            cache_shape_ok = cache_shape_ok and all(
                layer['k'].shape[1:3] == (lm.cfg.n_kv_heads, lm.cfg.window_size - 1)
                and layer['v'].shape == layer['k'].shape for layer in cache['layers'])
    prompt = torch.randint(4, lm.cfg.vocab_size, (2, 13), device=device)
    changed = prompt.clone()
    changed[:, 7:] = (changed[:, 7:] + 1) % lm.cfg.vocab_size
    future_error = float((lm(changed)[:, :7] - lm(prompt)[:, :7]).abs().max().item())
    # Independent explicit attention formula; this dense matrix is a test only.
    attention = lm.blocks[0].attn
    width = min(lm.cfg.window_size + 3, lm.cfg.max_seq_len)
    h = torch.randn(1, width, lm.cfg.d_model, device=device)
    q = attention.query(h)
    k, v = attention.keys_values(h)
    repeats = lm.cfg.n_heads // lm.cfg.n_kv_heads
    k, v = k.repeat_interleave(repeats, 1), v.repeat_interleave(repeats, 1)
    distance = torch.arange(width, device=device)[:, None] - torch.arange(width, device=device)[None, :]
    allowed = (distance >= 0) & (distance < lm.cfg.window_size)
    scores = (q @ k.transpose(-2, -1)) / (attention.head_dim ** 0.5)
    weights = torch.softmax(scores.masked_fill(~allowed, -float('inf')), dim=-1)
    dense = attention.out_proj((weights @ v).transpose(1, 2).reshape(1, width, lm.cfg.d_model))
    tiled = attention.local(h, lm.cfg.window_size, lm.cfg.query_chunk_size)
    local_error = float((dense - tiled).abs().max().item())
    result = {"cache_max_abs_error": max_cache_error, "padding_max_abs_error": max_padding_error,
              "future_token_max_abs_error": future_error,
              "dense_vs_tiled_window_max_abs_error": local_error,
              "compact_bounded_cache": cache_shape_ok, "tested_prompt_widths": widths,
              "tolerance": tolerance,
              "passed": cache_shape_ok and max(max_cache_error, max_padding_error, future_error, local_error) < tolerance}
    lm.train(was_training)
    return result


def fitting_check(model_config, tokens, steps=250, sequence_length=256, seed=42):
    require_two_gpus()
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cfg = ModelConfig(**dict(model_config, dropout=0.0))
    lm = wrapped = model = optimizer = scaler = None
    try:
        lm = build_model(cfg).cuda()
        math_result = numerical_check(lm)
        print("Attention/cache checks:", math_result, flush=True)
        if not math_result["passed"]:
            raise ValueError("Numerical model checks failed; stop before training")
        wrapped = ModelWithLoss(lm)
        model = nn.DataParallel(wrapped, device_ids=[0, 1])
        optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.999), eps=1e-8)
        scaler = torch.amp.GradScaler("cuda", init_scale=16384.0)
        x = torch.from_numpy(np.asarray(tokens[:sequence_length], dtype=np.int64)[None].copy()).cuda().repeat(2, 1)
        y = torch.from_numpy(np.asarray(tokens[1:sequence_length+1], dtype=np.int64)[None].copy()).cuda().repeat(2, 1)
        p = sequence_length // 2 if cfg.architecture != "efficient" else 0
        prefixes = torch.full((2,), p, dtype=torch.long, device=x.device)
        rows, applied, skipped = [], 0, 0
        started = time.monotonic()
        model.train()
        while applied < steps:
            optimizer.zero_grad(set_to_none=True)
            loss = model(x, y, prefixes).mean()
            if not torch.isfinite(loss).item():
                raise FloatingPointError("Nonfinite fitting loss")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() < old_scale:
                skipped += 1
                if skipped > 20:
                    raise FloatingPointError("Excessive fitting-probe overflow")
                continue
            applied += 1
            if applied == 1 or applied % 25 == 0 or applied == steps:
                row = {"step": applied, "nll_before_update": float(loss.item()),
                       "gradient_norm_before_clip": float(norm.item())}
                rows.append(row)
                print("Fitting:", row, flush=True)
        model.eval()
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            logits, labels = logits_and_labels(lm, x[:1], y[:1], p)
            final_nll = float(torch.nn.functional.cross_entropy(logits.flatten(0, 1), labels.flatten()).item())
            accuracy = float((logits.argmax(-1) == labels).float().mean().item())
        return {"type": "discarded_single_batch_fitting_check", "probe_config": dict(model_config, dropout=0.0),
                "sequence_length": sequence_length, "prefix_length": p, "applied_updates": applied,
                "skipped_updates": skipped, "final_nll": final_nll, "accuracy": accuracy,
                "numerical_checks": math_result, "passed": final_nll < 1.0 and accuracy >= 0.8,
                "history": rows, "elapsed_seconds": time.monotonic() - started}
    finally:
        lm = wrapped = model = optimizer = scaler = None
        gc.collect()
        torch.cuda.empty_cache()


def benchmark_microbatches(model_config, caches, sources, candidates=(2, 4, 8, 12),
                           effective_sequences=24, measured_updates=3, seed=42):
    require_two_gpus()
    cfg = ModelConfig(**model_config)
    assert cfg.max_seq_len == 2560
    prefixes = [1024, 1536, 2048] if cfg.architecture != "efficient" else [0, 0, 0]
    results = []
    for batch in candidates:
        assert batch % 2 == 0 and effective_sequences % batch == 0
        accumulation = effective_sequences // batch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        lm = wrapped = model = optimizer = scaler = None
        row = {"global_microbatch": batch, "accumulation_steps": accumulation,
               "effective_sequences": effective_sequences,
               "input_tokens_per_update": effective_sequences * cfg.max_seq_len,
               "tested_prefix_lengths": prefixes}
        try:
            lm = build_model(cfg).cuda()
            wrapped = ModelWithLoss(lm)
            model = nn.DataParallel(wrapped, device_ids=[0, 1])
            optimizer = torch.optim.Adam(model.parameters(), lr=1.5e-4)
            scaler = torch.amp.GradScaler("cuda", init_scale=16384.0)
            cursors, offset = initial_cursors(sources), 0
            orders = {s: block_order((len(caches[s])-1)//cfg.max_seq_len, seed, j, 0)
                      for j, s in enumerate(sources)}
            def update(prefix_length):
                nonlocal offset
                optimizer.zero_grad(set_to_none=True)
                for _ in range(accumulation):
                    xa, ya, offset, _ = mixed_batch(caches, cursors, sources, offset, batch, cfg.max_seq_len, seed, orders)
                    x, y = torch.from_numpy(xa).cuda(), torch.from_numpy(ya).cuda()
                    ps = torch.full((batch,), prefix_length, device=x.device, dtype=torch.long)
                    loss = model(x, y, ps).mean()
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError("Nonfinite benchmark loss")
                    scaler.scale(loss / accumulation).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                return scaler.get_scale() < old_scale
            model.train()
            warmup_skips = 0
            # Warm every boundary and allocate Adam state before measuring.
            for p in prefixes:
                while update(p):
                    warmup_skips += 1
                    if warmup_skips > 20:
                        raise FloatingPointError("Excessive benchmark overflow")
            for j in range(2):
                torch.cuda.synchronize(j)
                torch.cuda.reset_peak_memory_stats(j)
            started = time.monotonic()
            skips = sum(int(update(prefixes[j % len(prefixes)])) for j in range(measured_updates))
            for j in range(2):
                torch.cuda.synchronize(j)
            elapsed = time.monotonic() - started
            mem = memory()
            headroom = all(m["peak_reserved_gib"] <= 0.90*m["total_gib"] for m in mem)
            row.update(status="ok", eligible=headroom and skips == 0, gpu_memory=mem,
                elapsed_seconds=elapsed, warmup_skipped_updates=warmup_skips, measured_skipped_updates=skips,
                input_tokens_per_second=measured_updates * row["input_tokens_per_update"] / elapsed)
        except torch.cuda.OutOfMemoryError:
            row.update(status="out_of_memory", eligible=False, gpu_memory=memory())
        except FloatingPointError as error:
            row.update(status="numerical_failure", eligible=False, reason=str(error))
        finally:
            lm = wrapped = model = optimizer = scaler = None
            gc.collect()
            torch.cuda.empty_cache()
        results.append(row)
        print("Benchmark:", row, flush=True)
    good = [r for r in results if r["eligible"]]
    if not good:
        raise RuntimeError("No microbatch passes the memory/numerical gate; send these results")
    chosen = max(good, key=lambda r: r["input_tokens_per_second"])
    return {"architecture": cfg.architecture, "results": results, "chosen": chosen,
            "estimated_300m_training_hours_without_validation": 300_000_000 / chosen["input_tokens_per_second"] / 3600}
'''
(SRC_DIR / 'architecture_probes.py').write_bytes(_module_text.encode('utf-8'))


12036

## 10. Incremental generation

Reuse attention keys and values, apply the selected sampler, and write completed batches atomically so interrupted generation can resume.


In [9]:
_module_text = r'''"""Manual sampling with per-layer KV reuse and atomic fixed-batch resume."""
import json
import math
import time
from pathlib import Path
import torch
import torch.nn.functional as F
from architecture_models import select_cache
from phase1_utils import atomic_json, checked_json, read_json, text_hash


def sample_next(logits, mode, temperature, top_k, top_p, generator):
    if mode == "greedy":
        return int(logits.argmax().item())
    if temperature <= 0:
        raise ValueError("Sampling temperature must be positive")
    values = logits.float() / temperature
    if mode == "top_k":
        values, indices = values.topk(min(top_k, values.numel()))
    elif mode == "top_p":
        values, indices = values.sort(descending=True)
        probs = torch.softmax(values, dim=-1)
        values[probs.cumsum(-1) - probs >= top_p] = -float("inf")
    else:
        raise ValueError("Unknown decoding mode")
    chosen = torch.multinomial(torch.softmax(values, dim=-1), 1, generator=generator)
    return int(indices[chosen].item())


def decoded_text(tokenizer, ids):
    raw = b"".join(tokenizer.vocab[i] for i in ids if i >= 4)
    return raw.decode("utf-8", errors="replace")


def apply_repetition_controls(scores, generated, penalty=1.0, ngram_size=0):
    """Generated tokens only; CTRL penalty and HF-style repeated-ngram blocking.

    References: https://arxiv.org/abs/1909.05858 and
    https://huggingface.co/docs/transformers/v4.47.1/en/internal/generation_utils
    """
    if generated and penalty != 1.0:
        seen = torch.tensor(sorted(set(generated)), device=scores.device, dtype=torch.long)
        values = scores[seen]
        # Lower both positive and negative logits, once per distinct token.
        scores[seen] = torch.where(values < 0, values * penalty, values / penalty)
    if ngram_size > 0 and len(generated) >= ngram_size - 1:
        prefix = generated[-(ngram_size - 1):] if ngram_size > 1 else []
        banned = [generated[start + ngram_size - 1]
                  for start in range(len(generated) - ngram_size + 1)
                  if generated[start:start + ngram_size - 1] == prefix]
        if banned:
            scores[banned] = -float("inf")
    return scores


@torch.no_grad()
def generate_predictions(model, tokenizer, prompts, output_dir, protocol, batch_size=4,
                         max_new_tokens=1024, target_words=128, seed=456):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()
    ids = list(prompts)
    penalty = protocol.get("repetition_penalty", 1.0)
    ngram_size = protocol.get("no_repeat_ngram_size", 0)
    if not math.isfinite(penalty) or penalty < 1.0:
        raise ValueError("Repetition penalty must be finite and at least 1")
    if not isinstance(ngram_size, int) or ngram_size < 0:
        raise ValueError("No-repeat ngram size must be a nonnegative integer")
    engine = "incremental_kv_v1" if penalty == 1.0 and ngram_size == 0 else "incremental_kv_v2_repetition_controls"
    frozen = {**protocol, "engine": engine, "architecture": model.cfg.architecture,
        "prompt_sha256": text_hash(json.dumps(prompts, sort_keys=True)), "prompt_id_order": ids,
        "batch_size": batch_size, "max_new_tokens": max_new_tokens, "target_words": target_words,
        "seed": seed, "context": model.cfg.max_seq_len, "torch_version": str(torch.__version__)}
    checked_json(output_dir / "protocol.json", frozen)
    progress_path = output_dir / "progress.json"
    rows = read_json(progress_path)["rows"] if progress_path.exists() else {}
    if set(rows) - set(ids):
        raise ValueError("Progress contains unknown prompt IDs")
    started = time.monotonic()
    for start in range(0, len(ids), batch_size):
        group = ids[start:start + batch_size]
        done = [key in rows for key in group]
        if all(done):
            continue
        if any(done):
            raise ValueError("Saved progress contains a partial batch")
        tokens = [tokenizer.encode(prompts[key]["text"]) for key in group]
        lengths = torch.tensor([len(t) for t in tokens], device=device, dtype=torch.long)
        extra = 0 if model.cfg.architecture == "efficient" else 1
        if any(not t or len(t) + extra > model.cfg.max_seq_len for t in tokens):
            raise ValueError("A prompt is empty or exceeds the model context; no silent cropping")
        x = torch.zeros(len(group), int(lengths.max().item()), dtype=torch.long, device=device)
        for j, t in enumerate(tokens):
            x[j, :len(t)] = torch.tensor(t, device=device)
        generated = [[] for _ in group]
        reasons = ["token_cap"] * len(group)
        generators = [torch.Generator(device=device).manual_seed(seed + int(text_hash(k)[:8], 16)) for k in group]
        active = list(range(len(group)))
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits, cache = model.start_generation(x, lengths)
        for iteration in range(max_new_tokens):
            retained, next_ids = [], []
            for row_number, j in enumerate(active):
                scores = logits[row_number].float().clone()
                apply_repetition_controls(scores, generated[j], penalty, ngram_size)
                scores[:3] = -float("inf")
                text = decoded_text(tokenizer, generated[j])
                if len(text.split()) < target_words:
                    scores[3] = -float("inf")
                token = sample_next(scores, protocol["mode"], protocol.get("temperature", 1.0),
                    protocol.get("top_k", 50), protocol.get("top_p", 0.95), generators[j])
                if token == 3:
                    reasons[j] = "eos_after_word_target"
                    continue
                generated[j].append(token)
                text = decoded_text(tokenizer, generated[j])
                if len(text.split()) > target_words or (len(text.split()) >= target_words and text[-1:].isspace()):
                    reasons[j] = "word_target"
                    continue
                if int(cache["next_positions"][row_number].item()) >= model.cfg.max_seq_len:
                    reasons[j] = "context_cap"
                    continue
                retained.append(row_number)
                next_ids.append(token)
            if not retained or iteration + 1 == max_new_tokens:
                break
            index = torch.tensor(retained, dtype=torch.long, device=device)
            if len(retained) != len(active):
                cache = select_cache(cache, index)
            active = [active[i] for i in retained]
            step_ids = torch.tensor(next_ids, dtype=torch.long, device=device)[:, None]
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits, cache = model.generation_step(step_ids, cache)
        for j, key in enumerate(group):
            words = decoded_text(tokenizer, generated[j]).split()
            text = " ".join(words[:target_words])
            rows[key] = {"text": text, "generated_tokens": len(generated[j]), "word_count": len(text.split()),
                "stop_reason": reasons[j], "prompt_tokens": len(tokens[j]), "prompt_tokens_dropped": 0,
                "rolling_context_steps": 0, "contains_utf8_replacement": "\ufffd" in text,
                "complete": reasons[j] in ("word_target", "eos_after_word_target") and len(words) >= target_words}
        atomic_json(progress_path, {"rows": rows})
        del logits, cache
        print("Generated", start + len(group), "/", len(ids), flush=True)
    predictions = {key: {"text": rows[key]["text"]} for key in ids}
    atomic_json(output_dir / "prediction.json", predictions)
    atomic_json(output_dir / "summary.json", {"prompts": len(ids),
        "incomplete_ids": [k for k in ids if not rows[k]["complete"]],
        "elapsed_seconds_this_invocation": time.monotonic() - started,
        "cropped_prompt_count": 0, "rolling_context_used_count": 0,
        "utf8_replacement_ids": [k for k in ids if rows[k]["contains_utf8_replacement"]],
        "mean_generated_tokens": sum(rows[k]["generated_tokens"] for k in ids) / max(len(ids), 1)})
    return predictions, rows


@torch.no_grad()
def continuation_nll(model, tokenizer, prompts, truth):
    """Score the SAME held-out continuation tokens given each complete prompt.

    A leading space connects the prompt's final word to the continuation. This
    is a conditional metric, separate from each training-cache validation loss.
    """
    model.eval()
    device = next(model.parameters()).device
    groups = {}
    for key, record in prompts.items():
        prefix = tokenizer.encode(record["text"])
        labels = tokenizer.encode(" " + truth[key]["text"])
        if not prefix or not labels or len(prefix) + len(labels) > model.cfg.max_seq_len:
            raise ValueError("Conditional-NLL example exceeds context or is empty")
        p = torch.tensor(prefix, device=device, dtype=torch.long)[None]
        target = torch.tensor(labels, device=device, dtype=torch.long)[None]
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            if model.cfg.architecture == "efficient":
                known = torch.cat([p, target[:, :-1]], dim=1)
                logits = model(known)[:, len(prefix)-1:]
            else:
                decoder = torch.cat([torch.full_like(target[:, :1], 2), target[:, :-1]], dim=1)
                logits = model(p, decoder)
            loss_sum = float(F.cross_entropy(logits.flatten(0, 1), target.flatten(), reduction="sum").item())
        if key.startswith("val_"):
            source = "supreme_court"
        else:
            source = key.rsplit("_", 1)[0]
        group = groups.setdefault(source, {"nll_sum": 0.0, "targets": 0, "examples": 0})
        group["nll_sum"] += loss_sum
        group["targets"] += len(labels)
        group["examples"] += 1
    for group in groups.values():
        group["nll"] = group["nll_sum"] / group["targets"]
    return {"definition": "Next-token NLL of the same reference continuations given complete prompts",
            "sources": groups, "macro_source_nll": sum(g["nll"] for g in groups.values()) / len(groups),
            "token_nll": sum(g["nll_sum"] for g in groups.values()) / sum(g["targets"] for g in groups.values())}
'''
(SRC_DIR / 'architecture_generation.py').write_bytes(_module_text.encode('utf-8'))


10418

## 11. Preparation, checkpoint and evaluation protocol

Set `PREP_DIR` to the extracted directory with `cache_manifest.json` and `tokenizer_state.pkl`. Set `RUN_DIR` to the extracted run containing `best_model.pt` and `sources/`. The evaluation ID includes the checkpoint hash and batch size. Resume only with the matching local generation protocol.


In [11]:
from phase1_utils import read_json, atomic_json, checked_json, sha256_file, write_source, make_bundle
from architecture_models import ModelConfig, build_model
from architecture_probes import numerical_check
from a1_tokenizer import A1BPETokenizer
from architecture_generation import generate_predictions, continuation_nll
import architecture_generation as generation_module
assert Path(generation_module.__file__).resolve() == (SRC_DIR / 'architecture_generation.py').resolve(), 'Restart the kernel before running the updated notebook'
PREPARATION_ID = 'prep_002_mixed16k'
RUN_ID = 'efficient_001_mixed217m'
EVALUATION_ID_OVERRIDE = None  # normally use the automatic checkpoint-hash ID
PREP_DIR = Path(r'C:\CS781_Phase1\artifacts\cs781_phase1\preparation\prep_002_mixed16k')
RUN_DIR = Path(r'C:\CS781_Phase1\artifacts\cs781_phase1\runs\efficient_001_mixed217m')
GENERATION_BATCH_SIZE = 4
MAX_NEW_TOKENS = 1024
GENERATION_SEED = 456

required_prep = ['cache_manifest.json', 'tokenizer_state.pkl', 'validation_prompts.json',
                 'validation_truth.json', 'validation_groups.json']
required_run = ['best_model.pt', 'sources/architecture_models.py']
missing = [str(PREP_DIR / name) for name in required_prep if not (PREP_DIR / name).is_file()]
missing += [str(RUN_DIR / name) for name in required_run if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError('Check the two folder paths in cell 11. Missing files:\n' + '\n'.join(missing))
manifest = read_json(PREP_DIR / 'cache_manifest.json')
best_path = RUN_DIR / 'best_model.pt'
best_sha = sha256_file(best_path)
EVALUATION_ID = EVALUATION_ID_OVERRIDE or f'eval_{RUN_ID}_{best_sha[:12]}_efficient_local_v1_bs{GENERATION_BATCH_SIZE}'
EVAL_DIR = ROOT / 'evaluations' / EVALUATION_ID
EVAL_DIR.mkdir(parents=True, exist_ok=True)
for name in ['phase1_utils.py', 'phase1_paths.py', 'a1_tokenizer.py', 'phase1_data.py', 'architecture_models.py', 'architecture_training.py', 'architecture_probes.py', 'architecture_generation.py']:
    write_source(EVAL_DIR / 'sources' / name, (SRC_DIR / name).read_bytes())
trained_model_bytes = (RUN_DIR / 'sources/architecture_models.py').read_bytes()
local_model_bytes = (SRC_DIR / 'architecture_models.py').read_bytes()
if trained_model_bytes != local_model_bytes:
    only_line_endings = trained_model_bytes.replace(b'\r\n', b'\n') == local_model_bytes.replace(b'\r\n', b'\n')
    raise ValueError('Model source bytes differ. Only line-ending differences: ' + str(only_line_endings)
                     + '. Rerun the corrected source-writing cells; do not remove this check.')
assert sha256_file(PREP_DIR / 'tokenizer_state.pkl') == manifest['tokenizer_state_sha256']
prompts = read_json(PREP_DIR / 'validation_prompts.json')
truths = read_json(PREP_DIR / 'validation_truth.json')
groups = read_json(PREP_DIR / 'validation_groups.json')
assert len(prompts) == 125 and set(prompts) == set(truths)
environment = {name: version(name) for name in ['torch', 'numpy', 'bert-score', 'transformers', 'huggingface-hub']}
environment.update(python=platform.python_version(), cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0))
from bert_score.utils import lang2model, model2layers
metric_model = lang2model['en']
metric_defaults = {'lang': 'en', 'model_type': metric_model, 'num_layers': model2layers[metric_model],
                   'batch_size': 8, 'idf': False, 'rescale_with_baseline': False}
protocol = {'evaluation_id': EVALUATION_ID, 'run_id': RUN_ID, 'best_checkpoint_sha256': best_sha,
    'tokenizer_signature': manifest['tokenizer_signature'],
    'validation_prompts_sha256': sha256_file(PREP_DIR / 'validation_prompts.json'),
    'validation_truth_sha256': sha256_file(PREP_DIR / 'validation_truth.json'),
    'model_source_sha256': sha256_file(SRC_DIR / 'architecture_models.py'),
    'generation_source_sha256': sha256_file(SRC_DIR / 'architecture_generation.py'),
    'generation_batch_size': GENERATION_BATCH_SIZE, 'max_new_tokens': MAX_NEW_TOKENS,
    'seed': GENERATION_SEED, 'environment': environment, 'metric_defaults': metric_defaults}
checked_json(EVAL_DIR / 'evaluation_protocol.json', protocol)
for name in ['validation_prompts.json', 'validation_truth.json', 'validation_groups.json']:
    write_source(EVAL_DIR / name, (PREP_DIR / name).read_bytes())
print('Checkpoint:', best_path, '| SHA256:', best_sha)
print('Environment:', environment)
print('Official evaluator defaults:', metric_defaults)

print('Preparation folder:', PREP_DIR)
print('Training folder:', RUN_DIR)
print('Local evaluation folder:', EVAL_DIR)
print('Generation batch size:', GENERATION_BATCH_SIZE)


Checkpoint: C:\CS781_Phase1\artifacts\cs781_phase1\runs\efficient_001_mixed217m\best_model.pt | SHA256: c3796086871a995a378c1a8ab8c7471adcf29792b630a535a7125316ff73ecf6
Environment: {'torch': '2.11.0+cu128', 'numpy': '2.1.3', 'bert-score': '0.3.13', 'transformers': '5.16.1', 'huggingface-hub': '1.29.0', 'python': '3.12.10', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Laptop GPU'}
Official evaluator defaults: {'lang': 'en', 'model_type': 'roberta-large', 'num_layers': 17, 'batch_size': 8, 'idf': False, 'rescale_with_baseline': False}
Preparation folder: C:\CS781_Phase1\artifacts\cs781_phase1\preparation\prep_002_mixed16k
Training folder: C:\CS781_Phase1\artifacts\cs781_phase1\runs\efficient_001_mixed217m
Local evaluation folder: C:\Users\ashu7\Downloads\efficient_local_evaluation_output\cs781_phase1\evaluations\eval_efficient_001_mixed217m_c3796086871a_efficient_local_v1_bs4
Generation batch size: 4


### Checkpoint inspection

Read the optimizer update count, trained input positions and validation NLL directly from the evaluation weights.


In [12]:
saved = torch.load(
    RUN_DIR / "best_model.pt",
    map_location="cpu",
    weights_only=False,
    mmap=True,
)

result = saved["validation_result"]
print("Checkpoint updates:", result["optimizer_updates"])
print("Checkpoint input tokens:", result["input_tokens_in_applied_updates"])
print("Checkpoint validation NLL:", result["validation_nll"])

assert result["optimizer_updates"] == 4883
assert result["input_tokens_in_applied_updates"] == 300_011_520
assert abs(result["validation_nll"] - 1.40708362680632) < 1e-6

del saved
gc.collect()

Checkpoint updates: 4883
Checkpoint input tokens: 300011520
Checkpoint validation NLL: 1.40708362680632


30

## 12. Held-out continuations and reference likelihood

Check cached decoding against a full forward pass and generate continuations for the 125 held-out prompts. The optional reference NLL conditions on the full prompt and scores the actual reference continuation. The training-cache NLL follows the model's pretraining objective.


In [13]:
DECODING_SETTINGS = {
    'top_p_095_t1_r12_ng8': {'mode': 'top_p', 'top_p': .95, 'temperature': 1.0,
        'repetition_penalty': 1.2, 'no_repeat_ngram_size': 8},
    'greedy_r12_ng8': {'mode': 'greedy', 'repetition_penalty': 1.2, 'no_repeat_ngram_size': 8},
    'top_p_095_t1': {'mode': 'top_p', 'top_p': .95, 'temperature': 1.0},
    'greedy': {'mode': 'greedy'},
    'top_p_095_t07': {'mode': 'top_p', 'top_p': .95, 'temperature': .7},
    'top_k_50_t1': {'mode': 'top_k', 'top_k': 50, 'temperature': 1.0}}
RUN_DECODING_NAMES = ['top_p_095_t1']  # add 'greedy' only if you want another validation comparison
COMPUTE_MATCHED_CONTINUATION_NLL = True
best = torch.load(best_path, map_location='cpu', weights_only=False)
assert best['tokenizer_signature'] == manifest['tokenizer_signature']
assert best['model_config']['architecture'] == 'efficient'
lm = build_model(ModelConfig(**best['model_config']))
assert 50_000_000 <= sum(parameter.numel() for parameter in lm.parameters()) <= 300_000_000
lm.load_state_dict(best['model_state'], strict=True); lm = lm.to(DEVICE).eval()
print('Selected best validation:', best['validation_result'])
atomic_json(EVAL_DIR / 'checkpoint_details.json', {'architecture': best['model_config']['architecture'], 'model_config': best['model_config'], 'best_validation': best['validation_result'], 'checkpoint_sha256': best_sha})
del best
cache_check = numerical_check(lm)
atomic_json(EVAL_DIR / 'cache_check.json', cache_check)
assert cache_check['passed'], 'Cached logits failed equivalence check'
with open(PREP_DIR / 'tokenizer_state.pkl', 'rb') as file:
    tok_saved = pickle.load(file)
assert tok_saved['signature'] == manifest['tokenizer_signature']
tokenizer = A1BPETokenizer(); tokenizer.__dict__.update(tok_saved['state'])
if COMPUTE_MATCHED_CONTINUATION_NLL and not (EVAL_DIR / 'continuation_nll.json').exists():
    atomic_json(EVAL_DIR / 'continuation_nll.json', continuation_nll(lm, tokenizer, prompts, truths))
if (EVAL_DIR / 'continuation_nll.json').exists():
    print('Matched continuation NLL:', read_json(EVAL_DIR / 'continuation_nll.json'))
all_predictions, all_diagnostics = {}, {}
for name in RUN_DECODING_NAMES:
    decode_protocol = {**DECODING_SETTINGS[name], 'best_checkpoint_sha256': best_sha,
                       'generation_source_sha256': protocol['generation_source_sha256']}
    predictions, diagnostics = generate_predictions(lm, tokenizer, prompts, EVAL_DIR / 'validation' / name,
        decode_protocol, batch_size=GENERATION_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
        target_words=128, seed=GENERATION_SEED)
    all_predictions[name], all_diagnostics[name] = predictions, diagnostics
    failed = [key for key, row in diagnostics.items() if not row['complete'] or row['word_count'] != 128]
    print(name, '| incomplete:', failed)
del lm
gc.collect(); torch.cuda.empty_cache()  # free generator VRAM before loading BERTScore


Selected best validation: {'type': 'validation', 'optimizer_updates': 4883, 'input_tokens_in_applied_updates': 300011520, 'targets_in_applied_updates': 300011520, 'validation_nll': 1.40708362680632, 'validation_perplexity': 4.084027471134323, 'validation_token_nll': 1.4091584487759519, 'sources': {'supreme_court': {'nll': 1.689381184610199, 'perplexity': 5.416128084499748, 'targets': 189363, 'input_positions': 189363}, 'high_courts': {'nll': 1.0097905795041318, 'perplexity': 2.745026090093528, 'targets': 182097, 'input_positions': 182097}, 'acts': {'nll': 1.3466532780988, 'perplexity': 3.8445373787508625, 'targets': 186020, 'input_positions': 186020}, 'tribunals': {'nll': 1.5825094650121492, 'perplexity': 4.867154452611468, 'targets': 180484, 'input_positions': 180484}}}
Matched continuation NLL: {'definition': 'Next-token NLL of the same reference continuations given complete prompts', 'sources': {'supreme_court': {'nll_sum': 24178.20639038086, 'targets': 14479, 'examples': 50, 'nll':

## 13. Course evaluator

Write the supplied `evaluate_phase1.py` verbatim. It computes BERTScore F1 with English RoBERTa-large and reports ROUGE-L F1 and BLEU-4.


In [14]:
module_source = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 1 predictions on your own held-out split.')
    ap.add_argument('--truth', required=True, help='Your held-out reference JSON.')
    ap.add_argument('--prediction', required=True, help='Your prediction JSON.')
    args=ap.parse_args()
    truth=load_json(args.truth); pred=load_json(args.prediction)
    validate(truth,'truth'); validate(pred,'prediction')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    ps=[pred[i]['text'].split() for i in truth]; rs=[truth[i]['text'].split() for i in truth]
    pred_texts=[pred[i]['text'].strip() for i in truth]; ref_texts=[truth[i]['text'].strip() for i in truth]
    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore=float(f.mean().item())
    rouge=sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in truth)/len(truth)
    b4=bleu4(ps,rs)
    print(f'Examples: {len(truth)}')
    print(f'BERTScore F1: {bertscore:.6f}')
    print(f'ROUGE-L F1:   {rouge:.6f}')
    print(f'BLEU-4:       {b4:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}', file=sys.stderr); sys.exit(1)
'''
(SRC_DIR / 'evaluate_phase1.py').write_bytes(module_source.encode('utf-8'))
print('Wrote exact UTF-8 bytes:', SRC_DIR / 'evaluate_phase1.py')


Wrote exact UTF-8 bytes: C:\Users\ashu7\Downloads\architecture_src_efficient_local_v1\evaluate_phase1.py


## 14. Generation scores and completion rates

Score the 100 balanced cases, the 50 Supreme Court anchors, and the per-source slices. Score each group with its original IDs and generated text when all predictions are nonempty, and track incomplete generations separately.


In [15]:
# Score every nonempty prediction as generated, including short outputs.
evaluator_path = SRC_DIR / 'evaluate_phase1.py'
write_source(EVAL_DIR / 'sources/evaluate_phase1.py', evaluator_path.read_bytes())
import subprocess, re
score_rows = []
scoring_groups = {'balanced_100': groups['balanced_100'], 'anchor_50': groups['anchor_50'],
                  **{f'source_{s}': ids for s, ids in groups['balanced_source_ids'].items()}}
for name in RUN_DECODING_NAMES:
    predictions = read_json(EVAL_DIR / 'validation' / name / 'prediction.json')
    diagnostics = read_json(EVAL_DIR / 'validation' / name / 'progress.json')['rows']
    for group_name, group_ids in scoring_groups.items():
        missing = [key for key in group_ids if key not in predictions or key not in truths or key not in diagnostics]
        if missing:
            raise ValueError(f'Missing held-out examples for {name}/{group_name}: {missing}')
        empty = [key for key in group_ids if not predictions[key]['text'].strip()]
        if empty:
            raise ValueError(f'Empty predictions cannot be scored: {empty}')
        failed = [key for key in group_ids if not diagnostics[key]['complete']
                  or len(predictions[key]['text'].split()) != 128]
        if failed:
            print('Scoring all examples unchanged, including incomplete outputs:', name, group_name, failed)
        score_dir = EVAL_DIR / 'scores' / name / group_name
        score_dir.mkdir(parents=True, exist_ok=True)
        atomic_json(score_dir / 'truth.json', {key: truths[key] for key in group_ids})
        atomic_json(score_dir / 'prediction.json', {key: predictions[key] for key in group_ids})
        score_protocol = {'prediction_sha256': sha256_file(score_dir / 'prediction.json'),
            'truth_sha256': sha256_file(score_dir / 'truth.json'), 'evaluator_sha256': sha256_file(evaluator_path),
            'environment': environment}
        checked_json(score_dir / 'protocol.json', score_protocol)
        result_path = score_dir / 'result.json'
        if result_path.exists():
            result = read_json(result_path)
        else:
            completed = subprocess.run([sys.executable, str(evaluator_path), '--truth', str(score_dir / 'truth.json'),
                '--prediction', str(score_dir / 'prediction.json')], capture_output=True, text=True,
                encoding='utf-8', errors='replace',
                env={**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'})
            (score_dir / 'stdout.txt').write_text(completed.stdout, encoding='utf-8')
            (score_dir / 'stderr.txt').write_text(completed.stderr, encoding='utf-8')
            if completed.returncode:
                raise RuntimeError(f'Evaluator failed for {name}/{group_name}: {completed.stderr[-4000:]}')
            result = {'decoding': name, 'group': group_name, 'examples': len(group_ids)}
            for label, key in [('BERTScore F1', 'bertscore_f1'), ('ROUGE-L F1', 'rouge_l_f1'), ('BLEU-4', 'bleu4')]:
                match = re.search(re.escape(label) + r':\s*([0-9.eE+-]+)', completed.stdout)
                if not match:
                    raise ValueError(f'Missing {label} in evaluator output')
                result[key] = float(match.group(1))
        result.update(incomplete_count=len(failed), incomplete_ids=failed,
                      completion_rate=(len(group_ids) - len(failed)) / len(group_ids),
                      scoring_policy='All group examples scored unchanged; no dropping or padding')
        atomic_json(result_path, result)
        score_rows.append(result)
        print(result)
atomic_json(EVAL_DIR / 'score_summary.json', score_rows)
import pandas as pd
display(pd.DataFrame(score_rows))


{'decoding': 'top_p_095_t1', 'group': 'balanced_100', 'examples': 100, 'bertscore_f1': 0.839827, 'rouge_l_f1': 0.244141, 'bleu4': 0.141416, 'incomplete_count': 0, 'incomplete_ids': [], 'completion_rate': 1.0, 'scoring_policy': 'All group examples scored unchanged; no dropping or padding'}
{'decoding': 'top_p_095_t1', 'group': 'anchor_50', 'examples': 50, 'bertscore_f1': 0.821651, 'rouge_l_f1': 0.1725, 'bleu4': 0.022895, 'incomplete_count': 0, 'incomplete_ids': [], 'completion_rate': 1.0, 'scoring_policy': 'All group examples scored unchanged; no dropping or padding'}
{'decoding': 'top_p_095_t1', 'group': 'source_supreme_court', 'examples': 25, 'bertscore_f1': 0.819568, 'rouge_l_f1': 0.168125, 'bleu4': 0.019206, 'incomplete_count': 0, 'incomplete_ids': [], 'completion_rate': 1.0, 'scoring_policy': 'All group examples scored unchanged; no dropping or padding'}
{'decoding': 'top_p_095_t1', 'group': 'source_high_courts', 'examples': 25, 'bertscore_f1': 0.883329, 'rouge_l_f1': 0.428125, 'bl

,decoding,group,examples,bertscore_f1,rouge_l_f1,bleu4,incomplete_count,incomplete_ids,completion_rate,scoring_policy
0,top_p_095_t1,balanced_100,100,0.839827,0.244141,0.141416,0,[],1.0,All group examples scored unchanged; no droppi...
1,top_p_095_t1,anchor_50,50,0.821651,0.172500,0.022895,0,[],1.0,All group examples scored unchanged; no droppi...
2,top_p_095_t1,source_supreme_court,25,0.819568,0.168125,0.019206,0,[],1.0,All group examples scored unchanged; no droppi...
3,top_p_095_t1,source_high_courts,25,0.883329,0.428125,0.366796,0,[],1.0,All group examples scored unchanged; no droppi...
4,top_p_095_t1,source_acts,25,0.821913,0.194375,0.059973,0,[],1.0,All group examples scored unchanged; no droppi...
5,top_p_095_t1,source_tribunals,25,0.834496,0.185938,0.039892,0,[],1.0,All group examples scored unchanged; no droppi...


## 15. Qualitative comparison

Inspect fixed examples next to their references. Keep their word counts, stop reasons and generated text with the evaluation.


In [16]:
sample_ids = ['val_0000', 'val_0025', 'val_0049', *[ids[0] for ids in groups['balanced_source_ids'].values()]]
sample_ids = list(dict.fromkeys(sample_ids))
samples = {name: {key: {'prompt': prompts[key]['text'], 'reference': truths[key]['text'],
    'generated': read_json(EVAL_DIR / 'validation' / name / 'prediction.json')[key]['text'],
    'diagnostics': read_json(EVAL_DIR / 'validation' / name / 'progress.json')['rows'][key]}
    for key in sample_ids} for name in RUN_DECODING_NAMES}
atomic_json(EVAL_DIR / 'qualitative_samples.json', samples)
for name in RUN_DECODING_NAMES:
    print('\nDECODING:', name)
    for key in sample_ids[:3]:
        print(key, '| GENERATED:', samples[name][key]['generated'])
        print('REFERENCE:', samples[name][key]['reference'])



DECODING: top_p_095_t1
val_0000 | GENERATED: valid in law. Bench the House is certainly suggested to the effect that in the administration of any decision of the House in the House were made by the House. All India personal of the House are produced in a copy of Part B Part B Part B Part B, Part C with respect to a proposal about the consideration which does not contain only publicity, than the resolution. These provisions have virtually weakened being stated in the Advocates letter of 31st December, 1888. An Assembly Select for considering the question of elections in a Province cases? Deff of the House in the House was posed in a General House on [Refiner] pointed out the most important criteria for the proposition that where it is by the context
REFERENCE: on the same footing as the publication of proceedings in a court of law. Wason v. Walter, (1868-69) L.R. 4 Q.B. 73, referred to. M. S. M. Sharma v. Sri Krishna Sinha, [1959] SUPP. 1 S.C.R. 806, distinguished. Dr. Suresh Chandra B

## 16. Official test export

Set `GENERATE_OFFICIAL_TEST` and select the evaluated decoding name for a new export. The check requires the full set of official IDs and 128-word continuations. The ZIP contains `prediction.json` at its root.


In [18]:
GENERATE_OFFICIAL_TEST = True
CHOSEN_DECODING_NAME = "top_p_095_t1"      # e.g. 'top_p_095_t1', after reviewing validation scores
if GENERATE_OFFICIAL_TEST:
    if CHOSEN_DECODING_NAME not in DECODING_SETTINGS:
        raise ValueError('Choose a validation-tested decoding name')
    score_file = EVAL_DIR / 'scores' / CHOSEN_DECODING_NAME / 'balanced_100/result.json'
    if not score_file.exists():
        raise ValueError('Selected setting does not have a valid held-out score')
    selected_score = read_json(score_file)
    selected_rows = read_json(EVAL_DIR / 'validation' / CHOSEN_DECODING_NAME / 'progress.json')['rows']
    selected_predictions = read_json(EVAL_DIR / 'validation' / CHOSEN_DECODING_NAME / 'prediction.json')
    validation_failures = [key for key in prompts if not selected_rows[key]['complete']
        or len(selected_predictions[key]['text'].split()) != 128]
    if validation_failures:
        raise ValueError(f'Selected validation decoding is incomplete: {validation_failures}. Review decoding before official submission generation.')
    official_prompts = read_json(PREP_DIR / 'official_test_prompts.json')
    best = torch.load(best_path, map_location='cpu', weights_only=False)
    lm = build_model(ModelConfig(**best['model_config']))
    lm.load_state_dict(best['model_state']); lm = lm.to(DEVICE).eval()
    del best
    test_dir = EVAL_DIR / 'official_test' / CHOSEN_DECODING_NAME
    decode_protocol = {**DECODING_SETTINGS[CHOSEN_DECODING_NAME], 'best_checkpoint_sha256': best_sha,
                       'generation_source_sha256': protocol['generation_source_sha256']}
    test_predictions, test_rows = generate_predictions(lm, tokenizer, official_prompts, test_dir,
        decode_protocol, batch_size=GENERATION_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
        target_words=128, seed=GENERATION_SEED)
    del lm
    gc.collect(); torch.cuda.empty_cache()
    assert set(test_predictions) == set(official_prompts)
    failures = [key for key, row in test_rows.items() if not row['complete'] or row['word_count'] != 128]
    if failures:
        raise ValueError(f'Incomplete official continuations: {failures}. No submission ZIP created.')
    import zipfile
    submission_path = OUTPUT_ROOT / f'{EVALUATION_ID}_{CHOSEN_DECODING_NAME}_submission.zip'
    with zipfile.ZipFile(submission_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(test_dir / 'prediction.json', 'prediction.json')
    print('Official test submission file:', submission_path)
else:
    print('Official test generation is disabled until you choose a validated decoding setting.')


Generated 4 / 1000
Generated 8 / 1000
Generated 12 / 1000
Generated 16 / 1000
Generated 20 / 1000
Generated 24 / 1000
Generated 28 / 1000
Generated 32 / 1000
Generated 36 / 1000
Generated 40 / 1000
Generated 44 / 1000
Generated 48 / 1000
Generated 52 / 1000
Generated 56 / 1000
Generated 60 / 1000
Generated 64 / 1000
Generated 68 / 1000
Generated 72 / 1000
Generated 76 / 1000
Generated 80 / 1000
Generated 84 / 1000
Generated 88 / 1000
Generated 92 / 1000
Generated 96 / 1000
Generated 100 / 1000
Generated 104 / 1000
Generated 108 / 1000
Generated 112 / 1000
Generated 116 / 1000
Generated 120 / 1000
Generated 124 / 1000
Generated 128 / 1000
Generated 132 / 1000
Generated 136 / 1000
Generated 140 / 1000
Generated 144 / 1000
Generated 148 / 1000
Generated 152 / 1000
Generated 156 / 1000
Generated 160 / 1000
Generated 164 / 1000
Generated 168 / 1000
Generated 172 / 1000
Generated 176 / 1000
Generated 180 / 1000
Generated 184 / 1000
Generated 188 / 1000
Generated 192 / 1000
Generated 196 / 10

## 17. Evaluation backup

Package the evaluation protocol, saved predictions, generation diagnostics, and score summaries.


In [17]:
backup_path = OUTPUT_ROOT / f'{EVALUATION_ID}_backup.zip'
make_bundle(EVAL_DIR, backup_path, OUTPUT_ROOT)
print('Local evaluation backup:', backup_path.resolve())
print('Open this folder in File Explorer:', OUTPUT_ROOT.resolve())


Local evaluation backup: C:\Users\ashu7\Downloads\efficient_local_evaluation_output\eval_efficient_001_mixed217m_c3796086871a_efficient_local_v1_bs4_backup.zip
Open this folder in File Explorer: C:\Users\ashu7\Downloads\efficient_local_evaluation_output


## References

The Phase 1 brief defines 512-word prompts, 128-word continuations and the official evaluation interface. The embedded course evaluator computes BERTScore F1, ROUGE-L F1 and BLEU-4. Nucleus sampling follows Holtzman et al.

- [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751)
- [BERTScore](https://arxiv.org/abs/1904.09675)
